# Bi$_2$Se$_3$ background peak fits

This notebook generates publication-ready detector-space, caked-space, ROI, line-profile, and $Q_r$-rod figures for the Bi$_2$Se$_3$ background peak fits. It loads a saved RA-SIM GUI state from the parameter cell, `RA_SIM_ALL_BACKGROUND_STATE`, or `~/.local/share/ra_sim/all.json`, uses the 5°, 10°, and 15° background images, applies the repository detector orientation, cakes the measured images, fits local 2D Gaussian-plus-plane peak models in $(2\theta,\phi)$, and writes PNG/PDF figure outputs.

Quick terminal rerun for a different saved GUI state:

```bash
python scripts/diagnostics/run_all_background_peak_fits.py path/to/gui_state.json
```

Each state writes to `artifacts/background_peak_fits/<state-stem>_state` by default. Override with `OUTPUT_DIR`, `RA_SIM_ALL_BACKGROUND_OUT_DIR`, or `--out-dir`.

The plotting cells use journal-style typography, embedded vector-safe fonts, colorblind-safe overlays, and sentence-case labels.


In [ ]:
# Parameters. Leave blank to use environment variables and repository defaults.
GUI_STATE_PATH = ""
OUTPUT_DIR = ""
RUN_NAME = ""
FIT_WORKERS_OVERRIDE = ""
NUMBA_WORKERS_OVERRIDE = ""


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# Keep BLAS/OpenMP libraries from oversubscribing when Python-level workers are used.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy.optimize import least_squares

try:
    from numba import njit, prange, set_num_threads, get_num_threads
    NUMBA_AVAILABLE = True
except Exception:  # Numba is optional; the notebook still runs without JIT acceleration.
    NUMBA_AVAILABLE = False

    def njit(*args, **kwargs):  # type: ignore[override]
        if args and callable(args[0]) and len(args) == 1 and not kwargs:
            return args[0]
        def decorator(func):
            return func
        return decorator

    prange = range  # type: ignore[assignment]

    def set_num_threads(_n: int) -> None:  # type: ignore[override]
        return None

    def get_num_threads() -> int:  # type: ignore[override]
        return 1

from ra_sim.fitting.diffuse_background import DiffuseBackgroundConfig, fit_diffuse_background_native
from ra_sim.fitting.rod_profiles import caked_field_to_gui_phi, qz_profile_from_caked_mask
from ra_sim.gui.background import apply_background_backend_orientation
from ra_sim.gui import controllers as gui_controllers
from ra_sim.gui import qr_cylinder_overlay as gui_qr_cylinder_overlay
from ra_sim.io.osc_reader import read_osc
from ra_sim.utils.calculations import IndexofRefraction
from ra_sim.simulation.exact_cake_portable import (
    FastAzimuthalIntegrator,
    build_cake_transform_bundle,
    caked_point_to_detector_pixel,
    detector_pixel_angular_maps,
    detector_two_theta_max_deg,
    prepare_gui_phi_display,
    raw_phi_to_gui_phi,
)

DEFAULT_STATE_PATH = Path.home() / ".local" / "share" / "ra_sim" / "all.json"


def _setting_text(local_name: str, env_name: str, default: object = "") -> str:
    local_value = globals().get(local_name, "")
    if local_value is not None and str(local_value).strip():
        return str(local_value).strip()
    return str(os.environ.get(env_name, default)).strip()


def _safe_run_name(value: object) -> str:
    raw = str(value).strip().replace("\\", "/")
    text = Path(raw).stem if raw else "state"
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text).strip("._-")
    return text or "state"


def _positive_int_setting(local_name: str, env_name: str, default: int) -> int:
    text = _setting_text(local_name, env_name, "")
    try:
        value = int(text)
    except Exception:
        return int(default)
    return int(value) if value > 0 else int(default)


STATE_PATH = Path(
    _setting_text("GUI_STATE_PATH", "RA_SIM_ALL_BACKGROUND_STATE", DEFAULT_STATE_PATH)
).expanduser()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STATE_RUN_NAME = _safe_run_name(
    _setting_text("RUN_NAME", "RA_SIM_ALL_BACKGROUND_RUN_NAME", STATE_PATH.stem)
)
OUT_DIR = Path(
    _setting_text(
        "OUTPUT_DIR",
        "RA_SIM_ALL_BACKGROUND_OUT_DIR",
        ROOT / "artifacts" / "background_peak_fits" / f"{STATE_RUN_NAME}_state",
    )
).expanduser()
if not OUT_DIR.is_absolute():
    OUT_DIR = ROOT / OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = "Bi2Se3"
SAMPLE_LABEL = r"Bi$_2$Se$_3$"
TILT_BY_BACKGROUND = {0: 5, 1: 10, 2: 15}
EXCLUDED_PEAKS_BY_TILT = {
    (10, "0,0,9"),
    (15, "0,0,12"),
    (15, "-1,0,5"),
    (5, "-1,0,7"),
}
FIGURE7A_STEM = "figure7a_bi2se3_detector_peak_fits"
FIGURE7C_STEM = "figure7c_bi2se3_line_profile_fits"
MANUSCRIPT_DPI = 600

PIXEL_SIZE_M = 1.0e-4
WAVELENGTH_M = 1.54e-10
NPT_RADIAL = 1200
NPT_AZIMUTH = 720
THETA_HALF_WINDOW_DEG = 1.8
PHI_HALF_WINDOW_DEG = 6.0
CENTER_THETA_BOUND_DEG = 0.55
CENTER_PHI_BOUND_DEG = 1.6
DETECTOR_RENDER_SIGMA_RADIUS = 8.0
DETECTOR_RENDER_CHUNK_ROWS = 256
GAUSSIAN_TAIL_DISTANCE_WEIGHT = 1.25
GAUSSIAN_CORE_SIGNAL_DOWNSCALE = 0.06
GAUSSIAN_TAIL_OVERPREDICTION_START = 0.55
GAUSSIAN_TAIL_OVERPREDICTION_WEIGHT = 1.75
ROD_PROFILE_QZ_BINS = 96
ROD_PROFILE_STEM = "figure7_bi2se3_qr_rod_qz_profiles"
ROD_PROFILE_REGION_STEM = f"{ROD_PROFILE_STEM}_caked_selected_q_regions_5deg"
ROD_PROFILE_MAX_TWO_THETA_DEG = 60.0
CAKED_FIGURE_INTENSITY_MODE = "density"

CPU_COUNT = max(1, os.cpu_count() or 1)
FIT_WORKERS = max(1, min(_positive_int_setting("FIT_WORKERS_OVERRIDE", "BACKGROUND_FIT_WORKERS", CPU_COUNT), CPU_COUNT))
NUMBA_WORKERS = max(1, min(_positive_int_setting("NUMBA_WORKERS_OVERRIDE", "NUMBA_NUM_THREADS", CPU_COUNT), CPU_COUNT))
if NUMBA_AVAILABLE:
    try:
        set_num_threads(NUMBA_WORKERS)
    except Exception:
        pass

BACKGROUND_SUBTRACTION_ENABLED = False
BACKGROUND_SUBTRACTION_CONFIG = DiffuseBackgroundConfig(
    enabled=BACKGROUND_SUBTRACTION_ENABLED,
    mode="radial",
    apply_to_fit=True,
    apply_to_display=False,
    scale=1.0,
    auto_scale=False,
    radial_quantile=0.35,
    radial_smooth_sigma_deg=0.50,
    peak_mask_sigma=4.0,
    peak_mask_radius_px=10.0,
    direct_beam_mask_radius_px=35.0,
)

# Journal figure style. STIX ships with Matplotlib and gives consistent serif/math text
# without requiring a local LaTeX installation.
JOURNAL_FULL_WIDTH_IN = 7.20
JOURNAL_ATLAS_WIDTH_IN = 9.20
JOURNAL_LINE_WIDTH_PT = 0.75

OKABE_ITO = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
    "yellow": "#F0E442",
    "black": "#000000",
}
JOURNAL_DATA_COLOR = "0.18"
JOURNAL_FIT_COLOR = OKABE_ITO["vermillion"]
JOURNAL_CONTOUR_COLOR = OKABE_ITO["blue"]
JOURNAL_CENTER_COLOR = OKABE_ITO["orange"]
JOURNAL_GRID_COLOR = "0.88"
JOURNAL_INTENSITY_CMAP = "magma"
JOURNAL_MODEL_CMAP = "cividis"
JOURNAL_DETECTOR_CMAP = "gray"
JOURNAL_REGION_COLORS = [
    OKABE_ITO["blue"],
    OKABE_ITO["orange"],
    OKABE_ITO["green"],
    OKABE_ITO["vermillion"],
    OKABE_ITO["purple"],
    OKABE_ITO["sky"],
]

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "DejaVu Serif", "Times New Roman"],
    "mathtext.fontset": "stix",
    "font.size": 8.0,
    "axes.titlesize": 8.2,
    "axes.labelsize": 8.0,
    "xtick.labelsize": 7.0,
    "ytick.labelsize": 7.0,
    "legend.fontsize": 7.0,
    "figure.titlesize": 9.0,
    "axes.linewidth": 0.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 2.6,
    "ytick.major.size": 2.6,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "legend.frameon": False,
    "legend.handlelength": 1.8,
    "image.interpolation": "nearest",
    "savefig.facecolor": "white",
    "savefig.dpi": MANUSCRIPT_DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

print(f"state={STATE_PATH}")
print(f"run_name={STATE_RUN_NAME}")
print(f"out={OUT_DIR}")
print(f"cpu_count={CPU_COUNT} fit_workers={FIT_WORKERS} numba={NUMBA_AVAILABLE} numba_threads={get_num_threads()}")

In [ ]:

def as_float(value: object, fallback: float = float("nan")) -> float:
    try:
        out = float(value)
    except Exception:
        return float(fallback)
    return out if np.isfinite(out) else float(fallback)


def caked_image_for_intensity_mode(
    image: object,
    *,
    caked_sum_signal: object | None = None,
    caked_sum_normalization: object | None = None,
    caked_count: object | None = None,
    intensity_mode: str = "density",
) -> np.ndarray:
    base = np.asarray(image, dtype=np.float64)
    if base.ndim != 2:
        raise RuntimeError("caked image must be 2D")
    mode = "raw_sum" if str(intensity_mode) == "raw_sum" else "density"
    signal = None if caked_sum_signal is None else np.asarray(caked_sum_signal, dtype=np.float64)
    normalization = None if caked_sum_normalization is None else np.asarray(caked_sum_normalization, dtype=np.float64)
    if (signal is None) != (normalization is None):
        raise RuntimeError("caked sum_signal and sum_normalization fields must be paired")
    if signal is not None and signal.shape != base.shape:
        raise RuntimeError("caked sum_signal shape mismatch")
    if normalization is not None and normalization.shape != base.shape:
        raise RuntimeError("caked sum_normalization shape mismatch")

    if mode == "raw_sum":
        if signal is not None:
            return signal.copy()
        if caked_count is not None:
            count = np.asarray(caked_count, dtype=np.float64)
            if count.shape != base.shape:
                raise RuntimeError("caked count shape mismatch")
            return base * count
        return base.copy()

    if signal is None or normalization is None:
        return base.copy()
    density = np.full(base.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(signal) & np.isfinite(normalization) & (normalization > 0.0)
    density[valid] = signal[valid] / normalization[valid]
    return density


@njit(fastmath=True, nogil=True)
def _wrapped_delta_deg_scalar_numba(value: float, center: float) -> float:
    return ((value - center + 180.0) % 360.0) - 180.0


def wrapped_delta_deg(values: object, center: float) -> np.ndarray:
    return ((np.asarray(values, dtype=np.float64) - float(center) + 180.0) % 360.0) - 180.0


@njit(fastmath=True, nogil=True)
def _rotated_gaussian_value_numba(params: np.ndarray, theta: float, phi: float, peak_only: bool) -> float:
    amp = params[0]
    theta0 = params[1]
    phi0 = params[2]
    sigma_theta = max(params[3], 1.0e-12)
    sigma_phi = max(params[4], 1.0e-12)
    angle = params[5]
    dt = theta - theta0
    dp = _wrapped_delta_deg_scalar_numba(phi, phi0)
    cos_a = math.cos(angle)
    sin_a = math.sin(angle)
    u = cos_a * dt + sin_a * dp
    v = -sin_a * dt + cos_a * dp
    r2 = (u / sigma_theta) * (u / sigma_theta) + (v / sigma_phi) * (v / sigma_phi)
    peak = math.exp(-0.5 * r2)
    if peak_only:
        return amp * peak
    return params[6] + params[7] * dt + params[8] * dp + amp * peak


@njit(fastmath=True, nogil=True)
def _rotated_gaussian_value_from_matrix_numba(params_matrix: np.ndarray, peak_idx: int, theta: float, phi: float, peak_only: bool) -> float:
    amp = params_matrix[peak_idx, 0]
    theta0 = params_matrix[peak_idx, 1]
    phi0 = params_matrix[peak_idx, 2]
    sigma_theta = max(params_matrix[peak_idx, 3], 1.0e-12)
    sigma_phi = max(params_matrix[peak_idx, 4], 1.0e-12)
    angle = params_matrix[peak_idx, 5]
    dt = theta - theta0
    dp = _wrapped_delta_deg_scalar_numba(phi, phi0)
    cos_a = math.cos(angle)
    sin_a = math.sin(angle)
    u = cos_a * dt + sin_a * dp
    v = -sin_a * dt + cos_a * dp
    r2 = (u / sigma_theta) * (u / sigma_theta) + (v / sigma_phi) * (v / sigma_phi)
    peak = math.exp(-0.5 * r2)
    if peak_only:
        return amp * peak
    return params_matrix[peak_idx, 6] + params_matrix[peak_idx, 7] * dt + params_matrix[peak_idx, 8] * dp + amp * peak


@njit(fastmath=True, nogil=True)
def _rotated_gaussian_plane_points_numba(params: np.ndarray, theta_values: np.ndarray, phi_values: np.ndarray) -> np.ndarray:
    n = theta_values.size
    out = np.empty(n, dtype=np.float64)
    for idx in range(n):
        out[idx] = _rotated_gaussian_value_numba(params, theta_values[idx], phi_values[idx], False)
    return out


@njit(fastmath=True, nogil=True)
def _rotated_gaussian_peak_points_numba(params: np.ndarray, theta_values: np.ndarray, phi_values: np.ndarray) -> np.ndarray:
    n = theta_values.size
    out = np.empty(n, dtype=np.float64)
    for idx in range(n):
        out[idx] = _rotated_gaussian_value_numba(params, theta_values[idx], phi_values[idx], True)
    return out


@njit(fastmath=True, nogil=True)
def _rotated_gaussian_residual_points_numba(
    params: np.ndarray,
    theta_values: np.ndarray,
    phi_values: np.ndarray,
    y_values: np.ndarray,
    tail_weight: np.ndarray,
    denominator: np.ndarray,
    tail_overprediction_weight: np.ndarray,
) -> np.ndarray:
    n = theta_values.size
    out = np.empty(n, dtype=np.float64)
    for idx in range(n):
        model = _rotated_gaussian_value_numba(params, theta_values[idx], phi_values[idx], False)
        residual = model - y_values[idx]
        if residual > 0.0:
            residual *= tail_overprediction_weight[idx]
        out[idx] = tail_weight[idx] * residual / denominator[idx]
    return out


@njit(fastmath=True, nogil=True)
def _rotated_gaussian_grid_numba(params: np.ndarray, theta_axis: np.ndarray, phi_axis: np.ndarray, peak_only: bool) -> np.ndarray:
    n_phi = phi_axis.size
    n_theta = theta_axis.size
    out = np.empty((n_phi, n_theta), dtype=np.float64)
    for row in range(n_phi):
        phi = phi_axis[row]
        for col in range(n_theta):
            out[row, col] = _rotated_gaussian_value_numba(params, theta_axis[col], phi, peak_only)
    return out


@njit(parallel=True, nogil=True)
def _render_detector_peak_model_numba(
    theta_map: np.ndarray,
    phi_map: np.ndarray,
    params_matrix: np.ndarray,
    theta_radii: np.ndarray,
    phi_radii: np.ndarray,
) -> np.ndarray:
    height, width = theta_map.shape
    n_peaks = params_matrix.shape[0]
    out = np.zeros((height, width), dtype=np.float32)
    for row in prange(height):
        for col in range(width):
            theta = theta_map[row, col]
            phi = phi_map[row, col]
            if not (np.isfinite(theta) and np.isfinite(phi)):
                continue
            acc = 0.0
            for peak_idx in range(n_peaks):
                theta0 = params_matrix[peak_idx, 1]
                phi0 = params_matrix[peak_idx, 2]
                if abs(theta - theta0) <= theta_radii[peak_idx] and abs(_wrapped_delta_deg_scalar_numba(phi, phi0)) <= phi_radii[peak_idx]:
                    acc += _rotated_gaussian_value_from_matrix_numba(params_matrix, peak_idx, theta, phi, True)
            out[row, col] = np.float32(acc)
    return out


@njit(fastmath=True, nogil=True)
def _gaussian_lorentzian_profile_points_numba(params: np.ndarray, x_values: np.ndarray) -> np.ndarray:
    amp = params[0]
    center = params[1]
    sigma_g = max(params[2], 1.0e-12)
    gamma_l = max(params[3], 1.0e-12)
    gaussian_fraction = params[4]
    if gaussian_fraction < 0.0:
        gaussian_fraction = 0.0
    elif gaussian_fraction > 1.0:
        gaussian_fraction = 1.0
    baseline = params[5]
    slope = params[6]
    out = np.empty(x_values.size, dtype=np.float64)
    for idx in range(x_values.size):
        dx = x_values[idx] - center
        gaussian = math.exp(-0.5 * (dx / sigma_g) * (dx / sigma_g))
        lorentzian = 1.0 / (1.0 + (dx / gamma_l) * (dx / gamma_l))
        out[idx] = baseline + slope * dx + amp * (gaussian_fraction * gaussian + (1.0 - gaussian_fraction) * lorentzian)
    return out


@njit(fastmath=True, nogil=True)
def _gaussian_lorentzian_residual_numba(
    params: np.ndarray,
    x_values: np.ndarray,
    y_values: np.ndarray,
    tail_weight: np.ndarray,
    denominator: np.ndarray,
) -> np.ndarray:
    model = _gaussian_lorentzian_profile_points_numba(params, x_values)
    out = np.empty(x_values.size, dtype=np.float64)
    for idx in range(x_values.size):
        out[idx] = tail_weight[idx] * (model[idx] - y_values[idx]) / denominator[idx]
    return out


def rotated_gaussian_plane(params: np.ndarray, theta_grid: np.ndarray, phi_grid: np.ndarray) -> np.ndarray:
    theta_arr = np.asarray(theta_grid, dtype=np.float64)
    phi_arr = np.asarray(phi_grid, dtype=np.float64)
    if theta_arr.shape == phi_arr.shape:
        out = _rotated_gaussian_plane_points_numba(
            np.asarray(params, dtype=np.float64),
            np.ascontiguousarray(theta_arr.ravel()),
            np.ascontiguousarray(phi_arr.ravel()),
        )
        return out.reshape(theta_arr.shape)
    theta_b, phi_b = np.broadcast_arrays(theta_arr, phi_arr)
    out = _rotated_gaussian_plane_points_numba(
        np.asarray(params, dtype=np.float64),
        np.ascontiguousarray(theta_b.ravel()),
        np.ascontiguousarray(phi_b.ravel()),
    )
    return out.reshape(theta_b.shape)


def rotated_gaussian_peak_only(params: np.ndarray, theta_grid: np.ndarray, phi_grid: np.ndarray) -> np.ndarray:
    theta_arr = np.asarray(theta_grid, dtype=np.float64)
    phi_arr = np.asarray(phi_grid, dtype=np.float64)
    if theta_arr.shape == phi_arr.shape:
        out = _rotated_gaussian_peak_points_numba(
            np.asarray(params, dtype=np.float64),
            np.ascontiguousarray(theta_arr.ravel()),
            np.ascontiguousarray(phi_arr.ravel()),
        )
        return out.reshape(theta_arr.shape)
    theta_b, phi_b = np.broadcast_arrays(theta_arr, phi_arr)
    out = _rotated_gaussian_peak_points_numba(
        np.asarray(params, dtype=np.float64),
        np.ascontiguousarray(theta_b.ravel()),
        np.ascontiguousarray(phi_b.ravel()),
    )
    return out.reshape(theta_b.shape)


def gaussian_plane(params: np.ndarray, theta_grid: np.ndarray, phi_grid: np.ndarray) -> np.ndarray:
    return rotated_gaussian_plane(params, theta_grid, phi_grid)


def gaussian_peak_only(params: np.ndarray, theta_grid: np.ndarray, phi_grid: np.ndarray) -> np.ndarray:
    return rotated_gaussian_peak_only(params, theta_grid, phi_grid)


def render_detector_peak_model(theta_map: np.ndarray, phi_map: np.ndarray, fit_results: list[dict[str, object]]) -> np.ndarray:
    if not fit_results:
        return np.zeros(np.asarray(theta_map).shape, dtype=np.float32)
    params_matrix = np.ascontiguousarray([np.asarray(item["params"], dtype=np.float64) for item in fit_results], dtype=np.float64)
    theta_radii = np.empty(params_matrix.shape[0], dtype=np.float64)
    phi_radii = np.empty(params_matrix.shape[0], dtype=np.float64)
    for idx, p in enumerate(params_matrix):
        theta_radii[idx] = max(THETA_HALF_WINDOW_DEG, DETECTOR_RENDER_SIGMA_RADIUS * float(p[3]))
        phi_radii[idx] = max(PHI_HALF_WINDOW_DEG, DETECTOR_RENDER_SIGMA_RADIUS * float(p[4]))
    return _render_detector_peak_model_numba(
        np.ascontiguousarray(theta_map, dtype=np.float64),
        np.ascontiguousarray(phi_map, dtype=np.float64),
        params_matrix,
        theta_radii,
        phi_radii,
    )


def warm_numba_kernels() -> None:
    if not NUMBA_AVAILABLE:
        return
    try:
        p = np.array([10.0, 1.0, 0.0, 0.1, 0.5, 0.0, 1.0, 0.0, 0.0], dtype=np.float64)
        theta = np.linspace(0.8, 1.2, 8, dtype=np.float64)
        phi = np.linspace(-1.0, 1.0, 8, dtype=np.float64)
        y = np.ones(8, dtype=np.float64)
        _rotated_gaussian_grid_numba(p, theta, phi, False)
        _rotated_gaussian_residual_points_numba(p, theta, phi, y, y, y, y)
        _render_detector_peak_model_numba(theta.reshape(2, 4), phi.reshape(2, 4), p.reshape(1, -1), np.array([1.0]), np.array([1.0]))
        gp = np.array([1.0, 0.0, 0.4, 0.5, 0.5, 0.0, 0.0], dtype=np.float64)
        _gaussian_lorentzian_residual_numba(gp, phi, y, y, y)
    except Exception as exc:
        print(f"numba warm-up skipped: {exc}")


warm_numba_kernels()


def robust_display_limits(image: np.ndarray, low: float = 1.0, high: float = 99.7) -> tuple[float, float]:
    values = np.asarray(image, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0
    vmin, vmax = np.nanpercentile(values, [low, high])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = float(np.nanmin(values)), float(np.nanmax(values))
    if vmin == vmax:
        vmax = vmin + 1.0
    return float(vmin), float(vmax)


def label_from_entry(entry: dict[str, object]) -> str:
    hkl = entry.get("hkl")
    if isinstance(hkl, (list, tuple, np.ndarray)) and len(hkl) >= 3:
        return f"{int(hkl[0])},{int(hkl[1])},{int(hkl[2])}"
    return str(entry.get("label", "unknown"))


def branch_from_phi(phi: float) -> str:
    return "+" if float(phi) >= 0.0 else "-"


def compact_json_text(value: object) -> str:
    if value is None:
        return ""
    try:
        return json.dumps(value, separators=(",", ":"))
    except TypeError:
        return str(value)


def detector_xy_from_entry(entry: dict[str, object]) -> tuple[float, float] | None:
    for col_key, row_key in (
        ("background_detector_col", "background_detector_row"),
        ("detector_col", "detector_row"),
        ("x", "y"),
        ("fit_detector_col", "fit_detector_row"),
    ):
        col = as_float(entry.get(col_key))
        row = as_float(entry.get(row_key))
        if np.isfinite(col) and np.isfinite(row):
            return float(col), float(row)
    return None


def nearest_detector_angles_from_entry(entry: dict[str, object], theta_map: np.ndarray, phi_map: np.ndarray) -> tuple[float, float] | None:
    theta = as_float(entry.get("background_two_theta_deg"))
    phi = as_float(entry.get("background_phi_deg"))
    if np.isfinite(theta) and np.isfinite(phi):
        return float(theta), float(phi)
    xy = detector_xy_from_entry(entry)
    if xy is None:
        return None
    col, row = xy
    row_idx = int(np.clip(np.rint(row), 0, theta_map.shape[0] - 1))
    col_idx = int(np.clip(np.rint(col), 0, theta_map.shape[1] - 1))
    theta = float(theta_map[row_idx, col_idx])
    phi = float(phi_map[row_idx, col_idx])
    if not np.isfinite(theta) or not np.isfinite(phi):
        return None
    return theta, phi


def branch_sort_key(label: str) -> tuple[int, int, int, str]:
    try:
        parts = [int(part.strip()) for part in label.split(",")]
        if len(parts) >= 3:
            return abs(parts[0]) + abs(parts[1]), abs(parts[2]), parts[2], label
    except Exception:
        pass
    return 999, 999, 999, label


In [ ]:
state_doc = json.loads(STATE_PATH.read_text(encoding="utf-8"))
state = state_doc.get("state", state_doc)
files = state.get("files", {})
variables = state.get("variables", {}) if isinstance(state.get("variables"), dict) else {}
flags = state.get("flags", {}) if isinstance(state.get("flags"), dict) else {}
background_files = [str(path) for path in files.get("background_files", [])]
if not background_files:
    raise ValueError("all.json has no background files")
missing_tilts = sorted(set(range(len(background_files))) - set(TILT_BY_BACKGROUND))
if missing_tilts:
    raise ValueError(f"missing tilt labels for background indices: {missing_tilts}")

center_row_px = as_float(variables.get("center_x_var"), 0.0)
center_col_px = as_float(variables.get("center_y_var"), 0.0)
distance_m = as_float(variables.get("corto_detector_var"), 0.075)
backend_rotation_k = int(flags.get("background_backend_rotation_k", 3) or 0)
backend_flip_x = bool(flags.get("background_backend_flip_x", False))
backend_flip_y = bool(flags.get("background_backend_flip_y", False))
primary_cif_path = files.get("primary_cif_path")
try:
    n2_value = IndexofRefraction(WAVELENGTH_M)
except Exception:
    n2_value = complex(1.0, 0.0)

qr_rod_delta_qr = as_float(
    (state.get("analysis_range", {}) if isinstance(state.get("analysis_range"), dict) else {}).get("delta_qr"),
    as_float(variables.get("delta_qr_var"), 0.25),
)
rod_phi_samples = max(361, int(as_float(variables.get("rod_points_per_gz_var"), 721)))
psi_deg = as_float(variables.get("psi_var"), 0.0)


entries_by_bg: dict[int, list[dict[str, object]]] = {idx: [] for idx in range(len(background_files))}
for group in state.get("geometry", {}).get("manual_pairs", []) or []:
    if not isinstance(group, dict):
        continue
    bg_idx = int(group.get("background_index", -1))
    if bg_idx not in entries_by_bg:
        continue
    for entry in group.get("entries", []) or []:
        if not isinstance(entry, dict):
            continue
        theta = as_float(entry.get("background_two_theta_deg"))
        phi = as_float(entry.get("background_phi_deg"))
        has_caked_seed = np.isfinite(theta) and np.isfinite(phi)
        has_detector_seed = detector_xy_from_entry(entry) is not None
        if not (has_caked_seed or has_detector_seed):
            continue
        e = dict(entry)
        e["_background_index"] = bg_idx
        e["_background_name"] = Path(background_files[bg_idx]).name
        e["_sample_name"] = SAMPLE_NAME
        e["_tilt_deg"] = int(TILT_BY_BACKGROUND[bg_idx])
        e["_display_label"] = f"{SAMPLE_NAME} {TILT_BY_BACKGROUND[bg_idx]} deg"
        e["_label"] = label_from_entry(e)
        e["_branch"] = branch_from_phi(phi) if np.isfinite(phi) else None
        if (e["_tilt_deg"], e["_label"]) in EXCLUDED_PEAKS_BY_TILT:
            continue
        entries_by_bg[bg_idx].append(e)

expected_fit_count = sum(len(v) for v in entries_by_bg.values())
print(f"backgrounds={len(background_files)} points={expected_fit_count}")
print("excluded peaks:", ", ".join(f"{tilt} deg ({label})" for tilt, label in sorted(EXCLUDED_PEAKS_BY_TILT)))
print(f"orientation flip_x={backend_flip_x} flip_y={backend_flip_y} rotation_k={backend_rotation_k} (3/-1 = 90 deg CW)")
for idx, path in enumerate(background_files):
    print(f"{SAMPLE_NAME} {TILT_BY_BACKGROUND[idx]} deg: {Path(path).name} points={len(entries_by_bg[idx])}")

In [ ]:

def fit_one_peak(entry: dict[str, object], caked_image: np.ndarray, theta_axis: np.ndarray, phi_axis: np.ndarray, ai, transform_bundle, detector_shape) -> dict[str, object]:
    theta_seed = as_float(entry.get("_theta_seed_deg"), as_float(entry.get("background_two_theta_deg")))
    phi_seed = as_float(entry.get("_phi_seed_deg"), as_float(entry.get("background_phi_deg")))
    if not np.isfinite(theta_seed) or not np.isfinite(phi_seed):
        raise ValueError("missing peak seed angles")
    theta_mask = np.abs(theta_axis - theta_seed) <= THETA_HALF_WINDOW_DEG
    phi_mask = np.abs(wrapped_delta_deg(phi_axis, phi_seed)) <= PHI_HALF_WINDOW_DEG
    theta_idx = np.flatnonzero(theta_mask)
    phi_idx = np.flatnonzero(phi_mask)
    if theta_idx.size < 6 or phi_idx.size < 6:
        raise ValueError("ROI too small")

    roi = np.ascontiguousarray(caked_image[np.ix_(phi_idx, theta_idx)], dtype=np.float64)
    theta_vals = np.ascontiguousarray(theta_axis[theta_idx], dtype=np.float64)
    phi_vals = np.ascontiguousarray(phi_axis[phi_idx], dtype=np.float64)
    theta_grid, phi_grid = np.meshgrid(theta_vals, phi_vals)
    finite = np.isfinite(roi)
    if np.count_nonzero(finite) < 20:
        raise ValueError("too few finite ROI pixels")

    baseline0 = float(np.nanpercentile(roi[finite], 20.0))
    center_candidate = finite & (np.abs(theta_grid - theta_seed) <= CENTER_THETA_BOUND_DEG) & (np.abs(wrapped_delta_deg(phi_grid, phi_seed)) <= CENTER_PHI_BOUND_DEG)
    if np.any(center_candidate):
        peak_flat_index = int(np.nanargmax(np.where(center_candidate, roi, -np.inf)))
        peak_row, peak_col = np.unravel_index(peak_flat_index, roi.shape)
        theta0 = float(theta_vals[peak_col])
        phi0 = float(phi_vals[peak_row])
        amp0 = max(float(roi[peak_row, peak_col] - baseline0), 1.0)
    else:
        theta0 = theta_seed
        phi0 = phi_seed
        seed_distance = np.abs(theta_grid - theta_seed) + np.abs(wrapped_delta_deg(phi_grid, phi_seed))
        nearest_flat_index = int(np.nanargmin(np.where(finite, seed_distance, np.inf)))
        peak_row, peak_col = np.unravel_index(nearest_flat_index, roi.shape)
        amp0 = max(float(roi[peak_row, peak_col] - baseline0), 1.0)

    theta_step = float(np.nanmedian(np.diff(theta_axis))) if theta_axis.size > 1 else 0.05
    phi_step = float(np.nanmedian(np.diff(phi_axis))) if phi_axis.size > 1 else 0.5
    x0 = np.array([amp0, theta0, phi0, max(2.0 * abs(theta_step), 0.08), max(2.0 * abs(phi_step), 0.35), 0.0, baseline0, 0.0, 0.0], dtype=np.float64)
    lower = np.array([0.0, theta_seed - CENTER_THETA_BOUND_DEG, phi_seed - CENTER_PHI_BOUND_DEG, abs(theta_step) * 0.5, abs(phi_step) * 0.5, -math.pi / 2.0, -np.inf, -np.inf, -np.inf], dtype=np.float64)
    upper = np.array([max(amp0 * 60.0, amp0 + 1.0), theta_seed + CENTER_THETA_BOUND_DEG, phi_seed + CENTER_PHI_BOUND_DEG, THETA_HALF_WINDOW_DEG, PHI_HALF_WINDOW_DEG, math.pi / 2.0, np.inf, np.inf, np.inf], dtype=np.float64)
    x0 = np.minimum(np.maximum(x0, lower + 1.0e-9), upper - 1.0e-9)
    y = np.ascontiguousarray(roi[finite], dtype=np.float64)
    theta_fit = np.ascontiguousarray(theta_grid[finite], dtype=np.float64)
    phi_fit = np.ascontiguousarray(phi_grid[finite], dtype=np.float64)
    scale = max(float(np.nanpercentile(np.abs(y - np.nanmedian(y)), 75.0)), 1.0)

    tail_distance = np.sqrt(((theta_fit - theta0) / max(CENTER_THETA_BOUND_DEG, 1.0e-6)) ** 2 + (wrapped_delta_deg(phi_fit, phi0) / max(CENTER_PHI_BOUND_DEG, 1.0e-6)) ** 2)
    tail_weight = np.ascontiguousarray(1.0 + GAUSSIAN_TAIL_DISTANCE_WEIGHT * np.clip(tail_distance, 0.0, 2.0), dtype=np.float64)
    tail_excess = np.clip((tail_distance - GAUSSIAN_TAIL_OVERPREDICTION_START) / max(2.0 - GAUSSIAN_TAIL_OVERPREDICTION_START, 1.0e-6), 0.0, 1.0)
    tail_overprediction_weight = np.ascontiguousarray(1.0 + GAUSSIAN_TAIL_OVERPREDICTION_WEIGHT * tail_excess, dtype=np.float64)
    signal = np.clip(y - baseline0, 0.0, None)
    tail_scale = np.ascontiguousarray(scale + GAUSSIAN_CORE_SIGNAL_DOWNSCALE * signal, dtype=np.float64)

    def residual(params: np.ndarray) -> np.ndarray:
        return _rotated_gaussian_residual_points_numba(np.asarray(params, dtype=np.float64), theta_fit, phi_fit, y, tail_weight, tail_scale, tail_overprediction_weight)

    result = least_squares(
        residual,
        x0,
        bounds=(lower, upper),
        loss="soft_l1",
        f_scale=2.0,
        max_nfev=1000,
        x_scale=[max(amp0, 1.0), 0.2, 1.0, 0.2, 1.0, 1.0, max(abs(baseline0), 1.0), max(amp0, 1.0), max(amp0, 1.0)],
    )

    params = np.asarray(result.x, dtype=np.float64)
    fit = _rotated_gaussian_grid_numba(params, theta_vals, phi_vals, False)
    peak_fit = _rotated_gaussian_grid_numba(params, theta_vals, phi_vals, True)
    residual_image = roi - fit
    finite_resid = residual_image[finite]
    fit_col, fit_row = caked_point_to_detector_pixel(ai, detector_shape, theta_axis, phi_axis, float(params[1]), float(params[2]), transform_bundle=transform_bundle)
    return {
        "entry": dict(entry),
        "background_index": int(entry["_background_index"]),
        "background_name": str(entry["_background_name"]),
        "label": str(entry["_label"]),
        "branch": str(entry["_branch"]),
        "theta_seed_deg": float(theta_seed),
        "phi_seed_deg": float(phi_seed),
        "params": params,
        "theta_idx": theta_idx,
        "phi_idx": phi_idx,
        "theta_vals": theta_vals,
        "phi_vals": phi_vals,
        "roi": roi,
        "fit": fit,
        "peak_fit": peak_fit,
        "residual": residual_image,
        "success": bool(result.success),
        "message": str(result.message),
        "rmse": float(np.sqrt(np.nanmean(finite_resid ** 2))),
        "fit_detector_col": None if fit_col is None else float(fit_col),
        "fit_detector_row": None if fit_row is None else float(fit_row),
    }


In [ ]:
background_results = []
all_fit_results = []
all_table_rows = []

for bg_idx, bg_path_text in enumerate(background_files):
    bg_start = time.perf_counter()
    bg_path = Path(bg_path_text).expanduser()
    tilt_deg = int(TILT_BY_BACKGROUND[bg_idx])
    display_label = f"{SAMPLE_NAME} {tilt_deg} deg"
    native = np.asarray(read_osc(bg_path), dtype=np.float64)
    detector_image = apply_background_backend_orientation(native, flip_x=backend_flip_x, flip_y=backend_flip_y, rotation_k=backend_rotation_k)
    detector_image = np.asarray(detector_image, dtype=np.float64)

    ai = FastAzimuthalIntegrator(
        dist=float(distance_m),
        poni1=float(center_row_px) * PIXEL_SIZE_M,
        poni2=float(center_col_px) * PIXEL_SIZE_M,
        pixel1=PIXEL_SIZE_M,
        pixel2=PIXEL_SIZE_M,
        wavelength=WAVELENGTH_M,
    )
    tth_max = detector_two_theta_max_deg(detector_image.shape, ai.geometry)
    theta_map, raw_phi_map = detector_pixel_angular_maps(detector_image.shape, ai.geometry)
    phi_map = raw_phi_to_gui_phi(raw_phi_map)
    raw_detector_image = detector_image.copy()
    background_subtraction_result = None
    radial_background_model = np.zeros_like(detector_image, dtype=np.float64)
    if BACKGROUND_SUBTRACTION_ENABLED:
        background_subtraction_result = fit_diffuse_background_native(
            detector_image,
            two_theta_deg=theta_map,
            phi_deg=phi_map,
            config=BACKGROUND_SUBTRACTION_CONFIG,
            direct_beam_center_rc=(float(center_row_px), float(center_col_px)),
        )
        radial_background_model = np.asarray(
            background_subtraction_result.get("radial_model", background_subtraction_result.get("model")),
            dtype=np.float64,
        )
        detector_image = np.asarray(background_subtraction_result["corrected"], dtype=np.float64)

    caked_result = ai.integrate2d(detector_image, npt_rad=NPT_RADIAL, npt_azim=NPT_AZIMUTH, correctSolidAngle=True, method="lut", unit="2th_deg")
    caked_integrated_intensity, theta_axis, phi_axis = prepare_gui_phi_display(caked_result)
    raw_sum_signal = getattr(caked_result, "sum_signal", None)
    raw_sum_normalization = getattr(caked_result, "sum_normalization", None)
    raw_count = getattr(caked_result, "count", None)
    if (raw_sum_signal is None) != (raw_sum_normalization is None):
        raise RuntimeError("pyFAI caked sum_signal and sum_normalization fields must be paired")
    if raw_sum_signal is None:
        caked_sum_signal = None
        caked_sum_normalization = None
    else:
        caked_sum_signal, _, _ = caked_field_to_gui_phi(
            raw_sum_signal,
            caked_result.azimuthal_deg,
            caked_result.radial_deg,
        )
        caked_sum_normalization, _, _ = caked_field_to_gui_phi(
            raw_sum_normalization,
            caked_result.azimuthal_deg,
            caked_result.radial_deg,
        )
    if raw_count is None:
        caked_count = None
    else:
        caked_count, _, _ = caked_field_to_gui_phi(
            raw_count,
            caked_result.azimuthal_deg,
            caked_result.radial_deg,
        )
    caked_density_image = caked_image_for_intensity_mode(
        caked_integrated_intensity,
        caked_sum_signal=caked_sum_signal,
        caked_sum_normalization=caked_sum_normalization,
        caked_count=caked_count,
        intensity_mode="density",
    )
    caked_raw_sum_image = caked_image_for_intensity_mode(
        caked_integrated_intensity,
        caked_sum_signal=caked_sum_signal,
        caked_sum_normalization=caked_sum_normalization,
        caked_count=caked_count,
        intensity_mode="raw_sum",
    )
    caked_display_image = caked_image_for_intensity_mode(
        caked_integrated_intensity,
        caked_sum_signal=caked_sum_signal,
        caked_sum_normalization=caked_sum_normalization,
        caked_count=caked_count,
        intensity_mode=CAKED_FIGURE_INTENSITY_MODE,
    )
    caked_image = caked_density_image
    transform_bundle = build_cake_transform_bundle(ai, detector_image.shape, caked_result.radial_deg, caked_result.azimuthal_deg)
    caked_projection_context = {
        "detector_shape": tuple(detector_image.shape),
        "radial_axis": theta_axis,
        "azimuth_axis": phi_axis,
        "raw_azimuth_axis": caked_result.azimuthal_deg,
        "transform_bundle": transform_bundle,
    }
    qr_overlay_config = gui_qr_cylinder_overlay.build_qr_cylinder_overlay_render_config(
        render_in_caked_space=True,
        image_size=int(detector_image.shape[0]),
        display_rotate_k=0,
        center_col=float(center_col_px),
        center_row=float(center_row_px),
        distance_cor_to_detector=float(distance_m),
        gamma_deg=as_float(variables.get("gamma_var"), 0.0),
        Gamma_deg=as_float(variables.get("Gamma_var"), 0.0),
        chi_deg=as_float(variables.get("chi_var"), 0.0),
        psi_deg=psi_deg,
        psi_z_deg=as_float(variables.get("psi_z_var"), 0.0),
        zs=as_float(variables.get("zs_var"), 0.0),
        zb=as_float(variables.get("zb_var"), 0.0),
        theta_initial_deg=float(tilt_deg),
        cor_angle_deg=as_float(variables.get("cor_angle_var"), 0.0),
        pixel_size_m=PIXEL_SIZE_M,
        wavelength=WAVELENGTH_M * 1.0e10,
        n2=n2_value,
        phi_samples=rod_phi_samples,
    )

    prepared_entries = []
    seed_failures = []
    for entry in entries_by_bg[bg_idx]:
        seeded = nearest_detector_angles_from_entry(entry, theta_map, phi_map)
        if seeded is None:
            seed_failures.append({"entry": entry, "error": "missing detector/caked seed"})
            continue
        theta_seed, phi_seed = seeded
        prepared = dict(entry)
        prepared["_theta_seed_deg"] = float(theta_seed)
        prepared["_phi_seed_deg"] = float(phi_seed)
        prepared["_branch"] = branch_from_phi(phi_seed)
        prepared_entries.append(prepared)

    failures = list(seed_failures)
    fit_results_by_index: list[dict[str, object] | None] = [None] * len(prepared_entries)

    def _fit_prepared_entry(entry: dict[str, object]) -> dict[str, object]:
        return fit_one_peak(entry, caked_image, theta_axis, phi_axis, ai, transform_bundle, detector_image.shape)

    worker_count = min(FIT_WORKERS, max(1, len(prepared_entries)))
    if worker_count > 1:
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            future_to_index = {executor.submit(_fit_prepared_entry, entry): idx for idx, entry in enumerate(prepared_entries)}
            for future in as_completed(future_to_index):
                idx = int(future_to_index[future])
                entry = prepared_entries[idx]
                try:
                    fit_results_by_index[idx] = future.result()
                except Exception as exc:
                    failures.append({"entry": entry, "error": repr(exc)})
    else:
        for idx, entry in enumerate(prepared_entries):
            try:
                fit_results_by_index[idx] = _fit_prepared_entry(entry)
            except Exception as exc:
                failures.append({"entry": entry, "error": repr(exc)})

    fit_results = []
    for idx, item in enumerate(fit_results_by_index):
        if item is None:
            continue
        entry = prepared_entries[idx]
        fit_results.append(item)
        all_fit_results.append(item)
        p = item["params"]
        recorded_xy = detector_xy_from_entry(entry)
        all_table_rows.append({
            "sample_name": SAMPLE_NAME,
            "tilt_deg": tilt_deg,
            "background_index": bg_idx,
            "background_name": Path(bg_path).name,
            "label": item["label"],
            "branch": item["branch"],
            "q_group_key": compact_json_text(entry.get("q_group_key")),
            "branch_id": str(entry.get("branch_id", "")),
            "source_branch_index": str(entry.get("source_branch_index", "")),
            "selection_reason": str(entry.get("selection_reason", "")),
            "seed_two_theta_deg": item["theta_seed_deg"],
            "seed_phi_deg": item["phi_seed_deg"],
            "fit_two_theta_deg": float(p[1]),
            "fit_phi_deg": float(p[2]),
            "fit_sigma_two_theta_deg": float(p[3]),
            "fit_sigma_phi_deg": float(p[4]),
            "fit_angle_deg": float(np.rad2deg(p[5])),
            "fit_amplitude": float(p[0]),
            "fit_baseline": float(p[6]),
            "fit_model": "rotated_gaussian_plane",
            "fit_detector_col": item["fit_detector_col"],
            "fit_detector_row": item["fit_detector_row"],
            "recorded_detector_col": None if recorded_xy is None else float(recorded_xy[0]),
            "recorded_detector_row": None if recorded_xy is None else float(recorded_xy[1]),
            "rmse": item["rmse"],
            "success": item["success"],
        })

    caked_peak_model = np.zeros_like(caked_image, dtype=np.float64)
    for item in fit_results:
        caked_peak_model[np.ix_(item["phi_idx"], item["theta_idx"])] += item["peak_fit"]

    detector_peak_model = render_detector_peak_model(theta_map, phi_map, fit_results)

    np.save(OUT_DIR / f"background_{bg_idx:02d}_caked_peak_fit_model.npy", caked_peak_model)
    np.save(OUT_DIR / f"background_{bg_idx:02d}_detector_peak_fit_model.npy", detector_peak_model)
    background_results.append({
        "sample_name": SAMPLE_NAME,
        "tilt_deg": tilt_deg,
        "display_label": display_label,
        "background_index": bg_idx,
        "background_name": Path(bg_path).name,
        "detector_image": detector_image,
        "raw_detector_image": raw_detector_image,
        "radial_background_model": radial_background_model,
        "background_subtraction_result": background_subtraction_result,
        "caked_image": caked_density_image,
        "caked_integrated_intensity": caked_integrated_intensity,
        "caked_density_image": caked_density_image,
        "caked_raw_sum_image": caked_raw_sum_image,
        "caked_display_image": caked_display_image,
        "caked_intensity_mode": CAKED_FIGURE_INTENSITY_MODE,
        "caked_sum_signal": caked_sum_signal,
        "caked_sum_normalization": caked_sum_normalization,
        "caked_count": caked_count,
        "theta_axis": theta_axis,
        "phi_axis": phi_axis,
        "caked_peak_model": caked_peak_model,
        "detector_peak_model": detector_peak_model,
        "fit_results": fit_results,
        "failures": failures,
        "tth_max": tth_max,
        "caked_projection_context": caked_projection_context,
        "qr_overlay_config": qr_overlay_config,
        "raw_azimuth_axis": caked_result.azimuthal_deg,
        "transform_bundle": transform_bundle,
    })
    subtraction_label = "radial" if background_subtraction_result is not None else "raw"
    elapsed = time.perf_counter() - bg_start
    print(f"{display_label}: fit={len(fit_results)} fail={len(failures)} tth_max={tth_max:.3f} subtraction={subtraction_label} elapsed={elapsed:.2f}s workers={worker_count}")

fit_table = pd.DataFrame(all_table_rows)
fit_table_path = OUT_DIR / "all_background_peak_fit_table.csv"
fit_table.to_csv(fit_table_path, index=False)
print(f"table={fit_table_path}")
fit_table.head()

In [ ]:
def hkl_text(label: str) -> str:
    """Human-readable Miller-index label with a typographic minus sign."""
    return f"({str(label).replace('-', '−')})"

def tilt_text(value: int | float) -> str:
    return f"{int(value)}°"

def tilt_math(value: int | float) -> str:
    return rf"$\theta_i={int(value)}^\circ$"

def fit_model_vmax(model: np.ndarray, pct: float = 99.7) -> float:
    positive = np.asarray(model, dtype=np.float64)
    positive = positive[np.isfinite(positive) & (positive > 0.0)]
    if positive.size == 0:
        return 1.0
    vmax = float(np.nanpercentile(positive, pct))
    return vmax if np.isfinite(vmax) and vmax > 0 else 1.0

def detector_log_image(image: np.ndarray) -> np.ndarray:
    values = np.asarray(image, dtype=np.float64)
    offset = float(np.nanpercentile(values, 1.0))
    return np.log1p(np.clip(values - offset, 0.0, None))

def fit_log_image(model: np.ndarray) -> np.ndarray:
    return np.log1p(np.clip(np.asarray(model, dtype=np.float64), 0.0, None))

def shared_limits(images: list[np.ndarray], high: float = 99.7) -> tuple[float, float]:
    values = np.concatenate([np.asarray(image, dtype=np.float64)[np.isfinite(image)].ravel() for image in images])
    if values.size == 0:
        return 0.0, 1.0
    vmin = float(np.nanpercentile(values, 0.5))
    vmax = float(np.nanpercentile(values, high))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        return 0.0, 1.0
    return vmin, vmax

def detector_contour_levels(model: np.ndarray) -> np.ndarray:
    values = np.asarray(model, dtype=np.float64)
    values = values[np.isfinite(values) & (values > 0.0)]
    if values.size == 0:
        return np.array([], dtype=np.float64)
    levels = np.nanpercentile(values, [84.0, 93.0, 98.0])
    levels = np.unique(levels[np.isfinite(levels) & (levels > 0.0)])
    return levels

def overlay_fit_centers(ax, items: list[dict[str, object]], *, color: str = JOURNAL_CENTER_COLOR, size: float = 8.0) -> None:
    xs = [float(item["fit_detector_col"]) for item in items if item.get("fit_detector_col") is not None]
    ys = [float(item["fit_detector_row"]) for item in items if item.get("fit_detector_row") is not None]
    if xs and ys:
        ax.scatter(xs, ys, marker="x", s=size, c=color, linewidths=0.65, zorder=5)

def add_panel_label(ax, text: str, *, loc: str = "upper left") -> None:
    positions = {
        "upper left": (0.02, 0.98, "left", "top"),
        "upper right": (0.98, 0.98, "right", "top"),
        "lower left": (0.02, 0.02, "left", "bottom"),
    }
    x, y, ha, va = positions[loc]
    ax.text(
        x,
        y,
        text,
        transform=ax.transAxes,
        ha=ha,
        va=va,
        fontsize=7.7,
        color="white",
        bbox={"boxstyle": "round,pad=0.18", "facecolor": "black", "edgecolor": "none", "alpha": 0.62},
        zorder=6,
    )

def finish_axes(ax, *, grid: bool = False) -> None:
    ax.tick_params(which="both", direction="in", top=True, right=True, length=2.5, width=0.55, pad=2.0)
    if grid:
        ax.grid(True, color=JOURNAL_GRID_COLOR, linewidth=0.45, alpha=0.85)
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)

def save_manuscript_figure(fig, stem: str) -> tuple[Path, Path]:
    png_path = OUT_DIR / f"{stem}.png"
    pdf_path = OUT_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=MANUSCRIPT_DPI, bbox_inches="tight", pad_inches=0.025)
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.025)
    return png_path, pdf_path

def fit_lookup_by_key() -> dict[tuple[int, str, str], dict[str, object]]:
    lookup = {}
    for item in all_fit_results:
        key = (int(item["background_index"]), str(item["label"]), str(item["branch"]))
        lookup[key] = item
    return lookup

def get_fit_item(bg_idx: int, label: str, branch: str) -> dict[str, object]:
    lookup = fit_lookup_by_key()
    key = (int(bg_idx), str(label), str(branch))
    if key not in lookup:
        raise KeyError(f"missing fit item {key}")
    return lookup[key]

In [ ]:
fit_count = int(len(fit_table))
success_count = int(fit_table["success"].astype(bool).sum())
failure_count = sum(len(bg["failures"]) for bg in background_results)
if fit_count != expected_fit_count or success_count != expected_fit_count or failure_count != 0:
    print(f"fit rows={fit_count} successes={success_count} failures={failure_count}")
    for bg in background_results:
        if bg["failures"]:
            print(bg["display_label"], bg["failures"])
    raise RuntimeError("fit validation failed")
print(f"fit rows={fit_count}; successes={success_count}; failures={failure_count}; expected={expected_fit_count}")
print(f"labels: {SAMPLE_NAME}; tilts={', '.join(tilt_text(TILT_BY_BACKGROUND[i]) for i in sorted(TILT_BY_BACKGROUND))}")
print(f"orientation: apply_background_backend_orientation rotation_k={backend_rotation_k} (90 deg CW for this state)")

In [ ]:
# Complementary record of which peaks were used after exclusions.
def write_used_peaks_note() -> tuple[Path, Path]:
    rows = []
    for tilt in sorted(fit_table["tilt_deg"].unique()):
        sub = fit_table[fit_table["tilt_deg"] == tilt].copy()
        sub = sub.sort_values(["fit_two_theta_deg", "fit_phi_deg", "label", "branch"])
        for peak_number, (_, row) in enumerate(sub.iterrows(), start=1):
            rows.append({
                "sample_name": SAMPLE_NAME,
                "tilt_deg": int(tilt),
                "peak_number": peak_number,
                "hkl": str(row["label"]),
                "branch": str(row["branch"]),
                "q_group_key": str(row.get("q_group_key", "")),
                "branch_id": str(row.get("branch_id", "")),
                "source_branch_index": str(row.get("source_branch_index", "")),
                "fit_two_theta_deg": float(row["fit_two_theta_deg"]),
                "fit_phi_deg": float(row["fit_phi_deg"]),
                "fit_detector_col": float(row["fit_detector_col"]),
                "fit_detector_row": float(row["fit_detector_row"]),
            })
    used = pd.DataFrame(rows)
    csv_path = OUT_DIR / "figure7_bi2se3_used_peaks.csv"
    md_path = OUT_DIR / "figure7_bi2se3_used_peaks.md"
    used.to_csv(csv_path, index=False)

    lines = [
        "# Figure 7 Bi$_2$Se$_3$ used peaks",
        "",
        f"Source state: `{STATE_PATH}`",
        "",
        "The peaks below are the final filtered set used for the Bi$_2$Se$_3$ fitted-background figures and line-profile analysis.",
        "Peak numbers match the numbered markers in the per-tilt measured-background and fitted-peak-model figures.",
        "",
        "## Excluded peaks",
        "",
    ]
    for tilt, label in sorted(EXCLUDED_PEAKS_BY_TILT):
        lines.append(f"- {tilt_text(tilt)}: {hkl_text(label)}")
    lines.extend(["", "## Used peaks", ""])
    for tilt in sorted(used["tilt_deg"].unique()):
        sub = used[used["tilt_deg"] == tilt]
        lines.extend([
            f"### {tilt_text(tilt)}",
            "",
            r"| # | HKL | branch | source branch | fitted $2\theta$ (°) | fitted $\phi$ (°) | detector column (pixel) | detector row (pixel) |",
            "|---:|:---|:---:|:---:|---:|---:|---:|---:|",
        ])
        for _, row in sub.iterrows():
            lines.append(
                f"| {int(row['peak_number'])} | {hkl_text(row['hkl'])} | {row['branch']} | {row['source_branch_index']} | "
                f"{row['fit_two_theta_deg']:.4f} | {row['fit_phi_deg']:.4f} | "
                f"{row['fit_detector_col']:.1f} | {row['fit_detector_row']:.1f} |"
            )
        lines.append("")
    md_path.write_text("\n".join(lines), encoding="utf-8")
    return md_path, csv_path

used_peaks_md, used_peaks_csv = write_used_peaks_note()
print(f"saved={used_peaks_md}")
print(f"saved={used_peaks_csv}")

In [ ]:
# Figure 7a: detector-space measured images with fitted-peak contours and detector-space peak models.
ordered_backgrounds = sorted(background_results, key=lambda bg: int(bg["tilt_deg"]))
measured_log = [detector_log_image(bg["detector_image"]) for bg in ordered_backgrounds]
fit_log = [fit_log_image(bg["detector_peak_model"]) for bg in ordered_backgrounds]
measured_vmin, measured_vmax = shared_limits(measured_log, high=99.85)
fit_vmin, fit_vmax = shared_limits(fit_log, high=99.8)

fig, axes = plt.subplots(
    len(ordered_backgrounds),
    2,
    figsize=(JOURNAL_FULL_WIDTH_IN, max(2.45 * len(ordered_backgrounds), 7.0)),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
for row, bg in enumerate(ordered_backgrounds):
    detector_image = bg["detector_image"]
    detector_model = bg["detector_peak_model"]
    fit_items = bg["fit_results"]
    row_label = tilt_math(bg["tilt_deg"])

    ax = axes[row, 0]
    ax.imshow(detector_log_image(detector_image), origin="upper", cmap=JOURNAL_DETECTOR_CMAP, vmin=measured_vmin, vmax=measured_vmax)
    levels = detector_contour_levels(detector_model)
    if levels.size:
        ax.contour(detector_model, levels=levels, colors=JOURNAL_CONTOUR_COLOR, linewidths=0.50, origin="upper")
    overlay_fit_centers(ax, fit_items, color=JOURNAL_CENTER_COLOR, size=7.0)
    add_panel_label(ax, row_label)
    if row == 0:
        ax.set_title("Measured detector image with fitted contours")
    ax.set_ylabel("Detector row (pixel)")

    ax = axes[row, 1]
    ax.imshow(fit_log_image(detector_model), origin="upper", cmap=JOURNAL_MODEL_CMAP, vmin=fit_vmin, vmax=fit_vmax)
    overlay_fit_centers(ax, fit_items, color="white", size=6.0)
    add_panel_label(ax, row_label)
    if row == 0:
        ax.set_title("Fitted detector-space peak model")

for ax in axes[-1, :]:
    ax.set_xlabel("Detector column (pixel)")
for ax in axes.ravel():
    finish_axes(ax)

fig.suptitle(f"{SAMPLE_LABEL} detector-space peak fits", y=1.01)
fig7a_png, fig7a_pdf = save_manuscript_figure(fig, FIGURE7A_STEM)
plt.close(fig)
print(f"saved={fig7a_png}")
print(f"saved={fig7a_pdf}")
display(Image(filename=str(fig7a_png)))

In [ ]:
# Individual background-vs-fit figures for each Bi2Se3 tilt in caked and detector views.
def caked_log_image(image: np.ndarray) -> np.ndarray:
    values = np.asarray(image, dtype=np.float64)
    offset = float(np.nanpercentile(values, 1.0))
    return np.log1p(np.clip(values - offset, 0.0, None))

def peak_id_items(items: list[dict[str, object]]) -> list[tuple[int, dict[str, object]]]:
    ordered = sorted(items, key=lambda item: (float(item["params"][1]), float(item["params"][2]), str(item["label"]), str(item["branch"])))
    return list(enumerate(ordered, start=1))

def peak_id_label(item: dict[str, object]) -> str:
    return f"{hkl_text(str(item['label']))} {item['branch']}"





def annotate_peak_id(
    ax,
    peak_id: int,
    x: float,
    y: float,
    *,
    marker_color: str = JOURNAL_CONTOUR_COLOR,
    label_dx: float,
    label_dy: float,
) -> None:
    # Static double-stroked circle stays visible on bright yellow peaks and dark backgrounds.
    ax.scatter([x], [y], marker="o", s=50, facecolors="none", edgecolors="black", linewidths=1.15, zorder=6)
    ax.scatter([x], [y], marker="o", s=38, facecolors="none", edgecolors=marker_color, linewidths=1.00, zorder=7)
    ax.annotate(
        str(peak_id),
        xy=(x, y),
        xytext=(x + label_dx, y + label_dy),
        textcoords="data",
        ha="left",
        va="bottom",
        fontsize=5.4,
        color="black",
        fontweight="bold",
        bbox={"boxstyle": "round,pad=0.10", "facecolor": "white", "edgecolor": "0.15", "linewidth": 0.35, "alpha": 0.94},
        arrowprops={"arrowstyle": "->", "color": marker_color, "linewidth": 0.70, "shrinkA": 1.0, "shrinkB": 4.0},
        zorder=8,
    )

def overlay_caked_peak_ids(ax, numbered_items: list[tuple[int, dict[str, object]]], *, marker_color: str = JOURNAL_CONTOUR_COLOR, text_color: str = "black") -> None:
    label_dx = abs(ax.get_xlim()[1] - ax.get_xlim()[0]) * 0.012
    label_dy = abs(ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.018
    for peak_id, item in numbered_items:
        x = float(item["params"][1])
        y = float(item["params"][2])
        annotate_peak_id(ax, peak_id, x, y, marker_color=marker_color, label_dx=label_dx, label_dy=label_dy)

def overlay_detector_peak_ids(ax, numbered_items: list[tuple[int, dict[str, object]]], *, detector_model: np.ndarray, marker_color: str = JOURNAL_CONTOUR_COLOR, text_color: str = "black") -> None:
    label_dx = abs(ax.get_xlim()[1] - ax.get_xlim()[0]) * 0.018
    label_dy = -abs(ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.018
    for peak_id, item in numbered_items:
        x = item.get("fit_detector_col")
        y = item.get("fit_detector_row")
        if x is None or y is None:
            continue
        annotate_peak_id(ax, peak_id, float(x), float(y), marker_color=marker_color, label_dx=label_dx, label_dy=label_dy)

def add_peak_key(fig, numbered_items: list[tuple[int, dict[str, object]]]) -> None:
    lines = [f"{peak_id}: {peak_id_label(item)}" for peak_id, item in numbered_items]
    half = int(math.ceil(len(lines) / 2.0))
    key_text = "\n".join(lines[:half]) + "\n\n" + "\n".join(lines[half:])
    fig.text(
        1.015,
        0.50,
        key_text,
        ha="left",
        va="center",
        fontsize=6.0,
        family="serif",
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "0.65", "linewidth": 0.6},
    )

def save_background_fit_comparison(bg: dict[str, object]) -> tuple[Path, Path]:
    tilt = int(bg["tilt_deg"])
    fit_items = bg["fit_results"]
    numbered_items = peak_id_items(fit_items)
    theta_axis = bg["theta_axis"]
    phi_axis = bg["phi_axis"]
    caked_image = bg.get("caked_display_image", bg.get("caked_density_image", bg["caked_image"]))
    caked_mode_label = "raw accumulated intensity" if bg.get("caked_intensity_mode") == "raw_sum" else "intensity density"
    caked_model = bg["caked_peak_model"]
    detector_image = bg["detector_image"]
    detector_model = bg["detector_peak_model"]

    caked_extent = [float(theta_axis[0]), float(theta_axis[-1]), float(phi_axis[0]), float(phi_axis[-1])]
    caked_bg = caked_log_image(caked_image)
    caked_fit = fit_log_image(caked_model)
    caked_bg_vmin, caked_bg_vmax = robust_display_limits(caked_bg, low=1.0, high=99.65)
    caked_fit_vmin, caked_fit_vmax = robust_display_limits(caked_fit, low=0.0, high=99.7)
    detector_bg = detector_log_image(detector_image)
    detector_fit = fit_log_image(detector_model)
    detector_bg_vmin, detector_bg_vmax = robust_display_limits(detector_bg, low=1.0, high=99.85)
    detector_fit_vmin, detector_fit_vmax = robust_display_limits(detector_fit, low=0.0, high=99.8)

    fig, axes = plt.subplots(2, 2, figsize=(JOURNAL_FULL_WIDTH_IN, 6.9), constrained_layout=True)
    fig.suptitle(rf"{SAMPLE_LABEL}, $\theta_i={tilt}^\circ$: measured background and fitted peak model", y=1.01)

    ax = axes[0, 0]
    ax.imshow(caked_bg, extent=caked_extent, origin="lower", aspect="auto", cmap=JOURNAL_INTENSITY_CMAP, vmin=caked_bg_vmin, vmax=caked_bg_vmax)
    overlay_caked_peak_ids(ax, numbered_items, marker_color=JOURNAL_CONTOUR_COLOR)
    ax.set_title(f"Measured caked image ({caked_mode_label})")
    ax.set_xlabel(r"$2\theta$ (°)")
    ax.set_ylabel(r"$\phi$ (°)")

    ax = axes[0, 1]
    ax.imshow(caked_fit, extent=caked_extent, origin="lower", aspect="auto", cmap=JOURNAL_MODEL_CMAP, vmin=max(0.0, caked_fit_vmin), vmax=caked_fit_vmax)
    overlay_caked_peak_ids(ax, numbered_items, marker_color=JOURNAL_CONTOUR_COLOR)
    ax.set_title("Fitted caked-space peak model")
    ax.set_xlabel(r"$2\theta$ (°)")
    ax.set_ylabel(r"$\phi$ (°)")

    ax = axes[1, 0]
    ax.imshow(detector_bg, origin="upper", cmap=JOURNAL_DETECTOR_CMAP, vmin=detector_bg_vmin, vmax=detector_bg_vmax)
    overlay_detector_peak_ids(ax, numbered_items, detector_model=detector_model, marker_color=JOURNAL_CONTOUR_COLOR)
    ax.set_title("Measured detector image")
    ax.set_xlabel("Detector column (pixel)")
    ax.set_ylabel("Detector row (pixel)")

    ax = axes[1, 1]
    ax.imshow(detector_fit, origin="upper", cmap=JOURNAL_MODEL_CMAP, vmin=max(0.0, detector_fit_vmin), vmax=detector_fit_vmax)
    overlay_detector_peak_ids(ax, numbered_items, detector_model=detector_model, marker_color=JOURNAL_CONTOUR_COLOR)
    ax.set_title("Fitted detector-space peak model")
    ax.set_xlabel("Detector column (pixel)")
    ax.set_ylabel("Detector row (pixel)")

    for ax in axes.ravel():
        finish_axes(ax)
    add_peak_key(fig, numbered_items)

    stem = f"figure7_bi2se3_{tilt:02d}deg_background_vs_fit"
    png_path, pdf_path = save_manuscript_figure(fig, stem)
    plt.close(fig)
    return png_path, pdf_path

background_fit_comparison_paths = []
for bg in ordered_backgrounds:
    png_path, pdf_path = save_background_fit_comparison(bg)
    background_fit_comparison_paths.append((png_path, pdf_path))
    print(f"saved={png_path}")
    print(f"saved={pdf_path}")
    display(Image(filename=str(png_path)))


In [ ]:
# Paper ROI examples for specular peaks: measured caked background ROI beside fitted ROI image.
def save_specular_roi_examples() -> tuple[Path, Path]:
    requested_labels = ["0,0,3", "0,0,6"]
    available = {(int(item["background_index"]), str(item["label"]), str(item["branch"])): item for item in all_fit_results}
    specs = []
    for bg in ordered_backgrounds:
        bg_idx = int(bg["background_index"])
        for label in requested_labels:
            key = (bg_idx, label, "-")
            if key in available:
                specs.append((bg, available[key]))

    fig, axes = plt.subplots(
        len(specs),
        2,
        figsize=(JOURNAL_FULL_WIDTH_IN, max(1.45 * len(specs), 4.2)),
        squeeze=False,
        constrained_layout=True,
    )
    for row, (bg, item) in enumerate(specs):
        theta_vals = np.asarray(item["theta_vals"], dtype=np.float64)
        phi_vals = np.asarray(item["phi_vals"], dtype=np.float64)
        extent = [float(theta_vals[0]), float(theta_vals[-1]), float(phi_vals[0]), float(phi_vals[-1])]
        measured = np.asarray(item["roi"], dtype=np.float64)
        fitted = np.asarray(item["fit"], dtype=np.float64)
        finite_pair = np.concatenate([measured[np.isfinite(measured)].ravel(), fitted[np.isfinite(fitted)].ravel()])
        vmin, vmax = np.nanpercentile(finite_pair, [1.5, 99.3])
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            vmin, vmax = robust_display_limits(measured, low=1.5, high=99.3)
        row_label = f"{tilt_text(bg['tilt_deg'])} {hkl_text(str(item['label']))}"

        for col, (image, title) in enumerate([(measured, "Measured ROI"), (fitted, "Fitted ROI")]):
            ax = axes[row, col]
            ax.imshow(image, extent=extent, origin="lower", aspect="auto", cmap=JOURNAL_INTENSITY_CMAP, vmin=vmin, vmax=vmax)
            ax.plot(float(item["params"][1]), float(item["params"][2]), marker="x", color="white", markersize=4.0, mew=0.8)
            if row == 0:
                ax.set_title(title)
            if col == 0:
                ax.set_ylabel(f"{row_label}\n$\\phi$ (°)")
            else:
                ax.set_yticklabels([])
            if row == len(specs) - 1:
                ax.set_xlabel(r"$2\theta$ (°)")
            else:
                ax.set_xticklabels([])
            finish_axes(ax)

    fig.suptitle(f"{SAMPLE_LABEL} representative specular peak ROI fits", y=1.02)
    png_path, pdf_path = save_manuscript_figure(fig, "figure7_bi2se3_specular_roi_examples")
    plt.close(fig)
    return png_path, pdf_path

roi_examples_png, roi_examples_pdf = save_specular_roi_examples()
print(f"saved={roi_examples_png}")
print(f"saved={roi_examples_pdf}")
display(Image(filename=str(roi_examples_png)))

In [ ]:
# Figure 7c: local line-profile atlas comparing measured background to the caked-space fitted model.
FIGURE7C_PHI_PROFILE_FIT_STEM = f"{FIGURE7C_STEM}_phi_lorentzian_gaussian_fits"
PHI_GAUSSIAN_FWHM_FACTOR = 2.0 * math.sqrt(2.0 * math.log(2.0))


def raw_averaged_line_profile(item: dict[str, object], axis: str, band_half_width: int = 1) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    theta_vals = np.asarray(item["theta_vals"], dtype=np.float64)
    phi_vals = np.asarray(item["phi_vals"], dtype=np.float64)
    measured = np.asarray(item["roi"], dtype=np.float64)
    fitted = np.asarray(item["fit"], dtype=np.float64)
    theta0 = float(item["params"][1])
    phi0 = float(item["params"][2])

    phi_row = int(np.nanargmin(np.abs(wrapped_delta_deg(phi_vals, phi0))))
    theta_col = int(np.nanargmin(np.abs(theta_vals - theta0)))
    row_slice = slice(max(0, phi_row - band_half_width), min(measured.shape[0], phi_row + band_half_width + 1))
    col_slice = slice(max(0, theta_col - band_half_width), min(measured.shape[1], theta_col + band_half_width + 1))

    if axis == "theta":
        x = theta_vals - theta0
        measured_line = np.nanmean(measured[row_slice, :], axis=0)
        fitted_line = np.nanmean(fitted[row_slice, :], axis=0)
    elif axis == "phi":
        x = wrapped_delta_deg(phi_vals, phi0)
        measured_line = np.nanmean(measured[:, col_slice], axis=1)
        fitted_line = np.nanmean(fitted[:, col_slice], axis=1)
    else:
        raise ValueError(f"unknown profile axis {axis!r}")
    return x, measured_line, fitted_line


def normalize_line_pair(item: dict[str, object], measured_line: np.ndarray, fitted_line: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    finite = np.isfinite(measured_line) & np.isfinite(fitted_line)
    if not np.any(finite):
        return measured_line, fitted_line
    baseline = float(np.nanpercentile(measured_line[finite], 8.0))
    measured_span = float(np.nanpercentile(measured_line[finite], 99.0) - baseline)
    fitted_span = float(np.nanpercentile(fitted_line[finite], 99.0) - baseline)
    amp_span = abs(float(item["params"][0]))
    scale = max(measured_span, fitted_span, 0.25 * amp_span, 1.0)
    return (measured_line - baseline) / scale, (fitted_line - baseline) / scale


def averaged_line_profile(item: dict[str, object], axis: str, band_half_width: int = 1) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x, measured_line, fitted_line = raw_averaged_line_profile(item, axis=axis, band_half_width=band_half_width)
    measured_norm, fitted_norm = normalize_line_pair(item, measured_line, fitted_line)
    return x, measured_norm, fitted_norm


def gaussian_lorentzian_phi_profile(params: np.ndarray, x_values: np.ndarray) -> np.ndarray:
    return _gaussian_lorentzian_profile_points_numba(
        np.asarray(params, dtype=np.float64),
        np.ascontiguousarray(x_values, dtype=np.float64),
    )


def least_squares_parameter_stderr(result, n_points: int) -> np.ndarray:
    params = np.asarray(result.x, dtype=np.float64)
    stderr = np.full(params.shape, np.nan, dtype=np.float64)
    jac = np.asarray(result.jac, dtype=np.float64)
    if jac.ndim != 2 or jac.size == 0:
        return stderr
    dof = max(int(n_points) - int(params.size), 1)
    try:
        cov = np.linalg.pinv(jac.T @ jac) * (2.0 * float(result.cost) / float(dof))
    except np.linalg.LinAlgError:
        return stderr
    diag = np.diag(cov)
    valid = np.isfinite(diag) & (diag >= 0.0)
    stderr[valid] = np.sqrt(diag[valid])
    return stderr


def hkl_tuple_from_label(label: str) -> tuple[int, int, int] | None:
    try:
        parts = [int(part.strip()) for part in str(label).replace("(", "").replace(")", "").split(",")]
    except Exception:
        return None
    return tuple(parts[:3]) if len(parts) >= 3 else None


def is_hk_zero_peak(item: dict[str, object]) -> bool:
    hkl = hkl_tuple_from_label(str(item.get("label", "")))
    return bool(hkl is not None and hkl[0] == 0 and hkl[1] == 0)


def phi_profile_width_guess(x_values: np.ndarray, signal: np.ndarray) -> float:
    positive = np.asarray(signal, dtype=np.float64)
    x = np.asarray(x_values, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(positive) & (positive > 0.0)
    if np.count_nonzero(finite) >= 3:
        weights = positive[finite]
        center = float(np.sum(x[finite] * weights) / np.sum(weights))
        variance = float(np.sum(weights * (x[finite] - center) ** 2) / np.sum(weights))
        if np.isfinite(variance) and variance > 0.0:
            return float(np.clip(math.sqrt(variance), 0.12, 2.5))
    return 0.45


def fit_phi_gaussian_lorentzian_profile(x_values: np.ndarray, measured_line: np.ndarray) -> dict[str, object]:
    x = np.asarray(x_values, dtype=np.float64)
    y = np.asarray(measured_line, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    if np.count_nonzero(finite) < 8:
        raise ValueError("too few finite phi-profile points")
    x_fit = x[finite]
    y_fit = y[finite]
    baseline0 = float(np.nanpercentile(y_fit, 8.0))
    signal = np.clip(y_fit - baseline0, 0.0, None)
    peak_idx = int(np.nanargmax(signal if np.any(signal > 0.0) else y_fit))
    center0 = float(np.clip(x_fit[peak_idx], -CENTER_PHI_BOUND_DEG, CENTER_PHI_BOUND_DEG))
    amp0 = max(float(y_fit[peak_idx] - baseline0), 1.0)
    width0 = phi_profile_width_guess(x_fit, signal)
    x0 = np.array([amp0, center0, width0, width0, 0.50, baseline0, 0.0], dtype=np.float64)
    min_width = max(float(np.nanmedian(np.abs(np.diff(np.sort(np.unique(x_fit)))))) * 0.35, 0.025) if x_fit.size > 1 else 0.025
    max_width = max(PHI_HALF_WINDOW_DEG * 1.25, 2.5)
    lower = np.array([0.0, -CENTER_PHI_BOUND_DEG, min_width, min_width, 0.0, -np.inf, -np.inf], dtype=np.float64)
    upper = np.array([max(amp0 * 80.0, amp0 + 1.0), CENTER_PHI_BOUND_DEG, max_width, max_width, 1.0, np.inf, np.inf], dtype=np.float64)
    x0 = np.minimum(np.maximum(x0, lower + 1.0e-9), upper - 1.0e-9)
    noise = max(float(np.nanpercentile(np.abs(y_fit - np.nanmedian(y_fit)), 75.0)), 1.0)
    tail_weight = 1.0 + 0.45 * np.clip(np.abs(x_fit - center0) / max(CENTER_PHI_BOUND_DEG, 1.0e-6), 0.0, 2.5)
    local_signal = np.clip(y_fit - baseline0, 0.0, None)

    denominator = np.ascontiguousarray(noise + 0.08 * local_signal, dtype=np.float64)
    tail_weight = np.ascontiguousarray(tail_weight, dtype=np.float64)
    x_fit = np.ascontiguousarray(x_fit, dtype=np.float64)
    y_fit = np.ascontiguousarray(y_fit, dtype=np.float64)

    def residual(params: np.ndarray) -> np.ndarray:
        return _gaussian_lorentzian_residual_numba(np.asarray(params, dtype=np.float64), x_fit, y_fit, tail_weight, denominator)

    result = least_squares(
        residual,
        x0,
        bounds=(lower, upper),
        loss="soft_l1",
        f_scale=2.0,
        max_nfev=1000,
        x_scale=[max(amp0, 1.0), 0.5, 0.5, 0.5, 0.35, max(abs(baseline0), 1.0), max(amp0, 1.0)],
    )
    params = np.asarray(result.x, dtype=np.float64)
    stderr = least_squares_parameter_stderr(result, int(np.count_nonzero(finite)))
    model = gaussian_lorentzian_phi_profile(params, x)
    return {
        "success": bool(result.success),
        "message": str(result.message),
        "cost": float(result.cost),
        "params": params,
        "stderr": stderr,
        "model": model,
    }


def phi_profile_fit_report_row(bg: dict[str, object], item: dict[str, object], fit_payload: dict[str, object]) -> dict[str, object]:
    label = str(item["label"])
    hkl = hkl_tuple_from_label(label) or (None, None, None)
    params = np.asarray(fit_payload.get("params", np.full(7, np.nan)), dtype=np.float64)
    stderr = np.asarray(fit_payload.get("stderr", np.full(7, np.nan)), dtype=np.float64)
    gaussian_fraction = float(params[4]) if params.size > 4 else float("nan")
    gaussian_fraction_err = float(stderr[4]) if stderr.size > 4 else float("nan")
    entry = dict(item.get("entry", {}))
    return {
        "sample_name": SAMPLE_NAME,
        "tilt_deg": int(bg["tilt_deg"]),
        "background_index": int(bg["background_index"]),
        "peak_label": label,
        "h": hkl[0],
        "k": hkl[1],
        "l": hkl[2],
        "branch": str(item["branch"]),
        "q_group_key": entry.get("q_group_key"),
        "q_group_m": entry.get("q_group_m"),
        "q_group_qr": entry.get("q_group_qr"),
        "success": bool(fit_payload.get("success", False)),
        "message": str(fit_payload.get("message", "")),
        "phi_center_offset_deg": float(params[1]) if params.size > 1 else float("nan"),
        "phi_center_offset_deg_stderr": float(stderr[1]) if stderr.size > 1 else float("nan"),
        "gaussian_height_percent": 100.0 * gaussian_fraction,
        "gaussian_height_percent_stderr": 100.0 * gaussian_fraction_err,
        "lorentzian_height_percent": 100.0 * (1.0 - gaussian_fraction),
        "lorentzian_height_percent_stderr": 100.0 * gaussian_fraction_err,
        "gaussian_fwhm_phi_deg": PHI_GAUSSIAN_FWHM_FACTOR * float(params[2]) if params.size > 2 else float("nan"),
        "gaussian_fwhm_phi_deg_stderr": PHI_GAUSSIAN_FWHM_FACTOR * float(stderr[2]) if stderr.size > 2 else float("nan"),
        "lorentzian_fwhm_phi_deg": 2.0 * float(params[3]) if params.size > 3 else float("nan"),
        "lorentzian_fwhm_phi_deg_stderr": 2.0 * float(stderr[3]) if stderr.size > 3 else float("nan"),
        "fit_cost": float(fit_payload.get("cost", float("nan"))),
    }

ordered_backgrounds = sorted(background_results, key=lambda bg: int(bg["tilt_deg"]))
columns = [
    ("+", "theta", r"$+$ branch, $2\theta$ profiles"),
    ("+", "phi", r"$+$ branch, $\phi$ profiles"),
    ("-", "theta", r"$-$ branch, $2\theta$ profiles"),
    ("-", "phi", r"$-$ branch, $\phi$ profiles"),
]
axis_limits = {"theta": (-1.15, 1.15), "phi": (-3.9, 3.9)}
xlabels = {"theta": r"$\Delta 2\theta$ (°)", "phi": r"$\Delta\phi$ (°)"}
max_branch_count = max(
    (
        sum(1 for item in bg["fit_results"] if str(item["branch"]) == branch)
        for bg in ordered_backgrounds
        for branch in ("+", "-")
    ),
    default=1,
)
profile_row_height = max(3.0, 1.1 + 0.20 * max_branch_count)
phi_profile_fit_rows = []
phi_profile_fit_failures = []

def _empty_phi_profile_payload(message: str, measured_raw: np.ndarray) -> dict[str, object]:
    return {
        "success": False,
        "message": str(message),
        "cost": float("nan"),
        "params": np.full(7, np.nan, dtype=np.float64),
        "stderr": np.full(7, np.nan, dtype=np.float64),
        "model": np.full_like(np.asarray(measured_raw, dtype=np.float64), np.nan),
    }


def _fit_phi_profile_cache_item(bg: dict[str, object], item: dict[str, object]) -> tuple[int, dict[str, object]]:
    x, measured_raw, local_fit_raw = raw_averaged_line_profile(item, axis="phi", band_half_width=1)
    failure = None
    try:
        payload = fit_phi_gaussian_lorentzian_profile(x, measured_raw)
    except Exception as exc:
        payload = _empty_phi_profile_payload(str(exc), measured_raw)
        failure = (int(bg["tilt_deg"]), str(item["label"]), str(item["branch"]), str(exc))
    return id(item), {
        "x": x,
        "measured_raw": measured_raw,
        "local_fit_raw": local_fit_raw,
        "payload": payload,
        "failure": failure,
    }


phi_profile_cache: dict[int, dict[str, object]] = {}
phi_profile_targets = [(bg, item) for bg in ordered_backgrounds for item in bg["fit_results"]]
phi_worker_count = min(FIT_WORKERS, max(1, len(phi_profile_targets)))
if phi_worker_count > 1:
    with ThreadPoolExecutor(max_workers=phi_worker_count) as executor:
        future_to_item = {executor.submit(_fit_phi_profile_cache_item, bg, item): (bg, item) for bg, item in phi_profile_targets}
        for future in as_completed(future_to_item):
            bg, item = future_to_item[future]
            try:
                item_id, cached = future.result()
            except Exception as exc:
                x = np.array([], dtype=np.float64)
                measured_raw = np.array([], dtype=np.float64)
                cached = {
                    "x": x,
                    "measured_raw": measured_raw,
                    "local_fit_raw": measured_raw,
                    "payload": _empty_phi_profile_payload(str(exc), measured_raw),
                    "failure": (int(bg["tilt_deg"]), str(item["label"]), str(item["branch"]), str(exc)),
                }
                item_id = id(item)
            phi_profile_cache[item_id] = cached
            if cached["failure"] is not None:
                phi_profile_fit_failures.append(cached["failure"])
else:
    for bg, item in phi_profile_targets:
        item_id, cached = _fit_phi_profile_cache_item(bg, item)
        phi_profile_cache[item_id] = cached
        if cached["failure"] is not None:
            phi_profile_fit_failures.append(cached["failure"])
print(f"phi-profile fits cached={len(phi_profile_cache)} workers={phi_worker_count}")

fig, axes = plt.subplots(
    len(ordered_backgrounds),
    len(columns),
    figsize=(JOURNAL_ATLAS_WIDTH_IN, max(profile_row_height * len(ordered_backgrounds), 9.2)),
    constrained_layout=True,
)
for row, bg in enumerate(ordered_backgrounds):
    for col, (branch, axis, title) in enumerate(columns):
        ax = axes[row, col]
        items = [item for item in bg["fit_results"] if str(item["branch"]) == branch]
        items = sorted(items, key=lambda item: (float(item["params"][1]), branch_sort_key(str(item["label"]))))
        offsets = np.arange(len(items), dtype=np.float64) * 1.18

        for offset, item in zip(offsets, items):
            if axis == "phi":
                cached = phi_profile_cache[id(item)]
                x = np.asarray(cached["x"], dtype=np.float64)
                measured_raw = np.asarray(cached["measured_raw"], dtype=np.float64)
                local_fit_raw = np.asarray(cached["local_fit_raw"], dtype=np.float64)
                phi_fit_payload = cached["payload"]
                phi_fit_raw = np.asarray(phi_fit_payload.get("model"), dtype=np.float64)
                if phi_fit_raw.shape != np.asarray(measured_raw).shape or not np.any(np.isfinite(phi_fit_raw)):
                    phi_fit_raw = local_fit_raw
                measured_line, fitted_line = normalize_line_pair(item, measured_raw, phi_fit_raw)
                if not is_hk_zero_peak(item):
                    phi_profile_fit_rows.append(phi_profile_fit_report_row(bg, item, phi_fit_payload))
            else:
                x, measured_line, fitted_line = averaged_line_profile(item, axis=axis, band_half_width=1)
            mask = np.isfinite(x) & np.isfinite(measured_line) & np.isfinite(fitted_line)
            if not np.any(mask):
                continue
            ax.plot(x[mask], measured_line[mask] + offset, color=JOURNAL_DATA_COLOR, linewidth=0.72, alpha=0.90)
            ax.plot(x[mask], fitted_line[mask] + offset, color=JOURNAL_FIT_COLOR, linewidth=0.88, alpha=0.96)

        ax.set_xlim(*axis_limits[axis])
        ax.set_ylim(-0.35, (offsets[-1] + 1.05) if offsets.size else 1.0)
        ax.grid(True, axis="x", color=JOURNAL_GRID_COLOR, linewidth=0.45)
        ax.axvline(0.0, color="0.72", linewidth=0.45)
        if row == 0:
            ax.set_title(title)
        if row == len(ordered_backgrounds) - 1:
            ax.set_xlabel(xlabels[axis])
        if col in (0, 2):
            ax.set_yticks(offsets)
            ax.set_yticklabels([hkl_text(str(item["label"])) for item in items], fontsize=6.0)
        else:
            ax.set_yticks(offsets)
            ax.set_yticklabels([])
        if col == 0:
            ax.text(
                -0.34,
                0.5,
                tilt_math(bg["tilt_deg"]),
                transform=ax.transAxes,
                rotation=90,
                ha="center",
                va="center",
                fontsize=9,
            )
        ax.tick_params(axis="x", labelsize=6.5, length=2.0, width=0.45, direction="in", top=True)
        ax.tick_params(axis="y", length=0.0, width=0.0, pad=1.0)
        for spine in ax.spines.values():
            spine.set_linewidth(0.55)

legend_handles = [
    axes[0, 0].plot([], [], color=JOURNAL_DATA_COLOR, linewidth=0.9, label="Measured background")[0],
    axes[0, 0].plot([], [], color=JOURNAL_FIT_COLOR, linewidth=1.0, label="Fitted local peak model")[0],
]
fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.52, 1.015))
fig.suptitle(f"{SAMPLE_LABEL} local line-profile fits for recorded peaks", y=1.04)

phi_profile_fit_table = pd.DataFrame(phi_profile_fit_rows)
phi_profile_fit_csv = OUT_DIR / f"{FIGURE7C_PHI_PROFILE_FIT_STEM}.csv"
phi_profile_fit_summary_csv = OUT_DIR / f"{FIGURE7C_PHI_PROFILE_FIT_STEM}_summary.csv"
phi_profile_fit_note = OUT_DIR / f"{FIGURE7C_PHI_PROFILE_FIT_STEM}.md"
phi_profile_fit_table.to_csv(phi_profile_fit_csv, index=False)

summary_rows = []
metric_columns = [
    "gaussian_height_percent",
    "lorentzian_height_percent",
    "gaussian_fwhm_phi_deg",
    "lorentzian_fwhm_phi_deg",
]
if not phi_profile_fit_table.empty:
    good_phi_fits = phi_profile_fit_table[phi_profile_fit_table["success"].astype(bool)].copy()
    for (peak_label, branch), sub in good_phi_fits.groupby(["peak_label", "branch"], sort=True):
        row_summary = {
            "sample_name": SAMPLE_NAME,
            "peak_label": peak_label,
            "branch": branch,
            "h": sub["h"].iloc[0],
            "k": sub["k"].iloc[0],
            "l": sub["l"].iloc[0],
            "fit_count": int(len(sub)),
        }
        for metric in metric_columns:
            values = np.asarray(sub[metric], dtype=np.float64)
            values = values[np.isfinite(values)]
            row_summary[f"{metric}_mean"] = float(np.nanmean(values)) if values.size else float("nan")
            row_summary[f"{metric}_observed_sem"] = float(np.nanstd(values, ddof=1) / math.sqrt(values.size)) if values.size > 1 else float("nan")
            stderr_col = f"{metric}_stderr"
            if stderr_col in sub:
                stderr_values = np.asarray(sub[stderr_col], dtype=np.float64)
                stderr_values = stderr_values[np.isfinite(stderr_values)]
                row_summary[f"{metric}_fit_stderr_on_mean"] = float(math.sqrt(np.nansum(stderr_values ** 2)) / max(stderr_values.size, 1)) if stderr_values.size else float("nan")
        summary_rows.append(row_summary)
phi_profile_fit_summary = pd.DataFrame(summary_rows)
phi_profile_fit_summary.to_csv(phi_profile_fit_summary_csv, index=False)

phi_fit_note_lines = [
    f"# {SAMPLE_NAME} phi-profile Gaussian/Lorentzian fits",
    "",
    "Only non-H=K=0 peaks are reported.",
    "Each reported phi profile is fit to `baseline + slope*(phi-center) + amplitude*(g*Gaussian + (1-g)*Lorentzian)`.",
    "Gaussian/Lorentzian percentages are peak-height mixing percentages; FWHM values are in degrees of phi.",
    "Per-row error margins are one-standard-error estimates from the least-squares covariance.",
    "The summary CSV groups successful fits by peak label and branch; observed SEM is across all used tilts/instances.",
    "The 2theta profiles in Figure 7c still use the existing caked-space local peak model.",
    "",
    f"Detailed CSV: `{phi_profile_fit_csv.name}`",
    f"Summary CSV: `{phi_profile_fit_summary_csv.name}`",
    f"Reported rows: {len(phi_profile_fit_table)}",
    f"Successful rows: {int(phi_profile_fit_table['success'].sum()) if not phi_profile_fit_table.empty else 0}",
]
phi_profile_fit_note.write_text("\n".join(phi_fit_note_lines) + "\n", encoding="utf-8")
print(f"saved={phi_profile_fit_csv}")
print(f"saved={phi_profile_fit_summary_csv}")
print(f"saved={phi_profile_fit_note}")
if phi_profile_fit_failures:
    print("phi profile fit failures:")
    for failure in phi_profile_fit_failures:
        print(f"  tilt={failure[0]} label={failure[1]} branch={failure[2]} reason={failure[3]}")

fig7c_png, fig7c_pdf = save_manuscript_figure(fig, FIGURE7C_STEM)
plt.close(fig)
print(f"saved={fig7c_png}")
print(f"saved={fig7c_pdf}")
display(Image(filename=str(fig7c_png)))

In [ ]:
# $Q_r$-rod profiles from detector-space $Q_r$/$Q_z$ integration for the 5° background only.
import importlib
from ra_sim.simulation import exact_qspace_portable
from ra_sim.simulation.exact_cake import integrate_detector_to_cake_lut
try:
    from ra_sim.simulation import intersection_analysis as _intersection_analysis
    _intersection_analysis = importlib.reload(_intersection_analysis)
except Exception:
    _intersection_analysis = None
IntersectionGeometry = getattr(_intersection_analysis, "IntersectionGeometry", None)
detector_points_to_sample_qr_qz = getattr(_intersection_analysis, "detector_points_to_sample_qr_qz", None)
ROD_PROFILE_TILT_DEG = 5
ROD_PROFILE_MARKER_STEM = f"{ROD_PROFILE_STEM}_peak_markers_5deg"

def parse_hkl_label(label: str) -> tuple[int, int, int]:
    parts = [int(part.strip()) for part in str(label).split(",")]
    if len(parts) != 3:
        raise ValueError(f"invalid HKL label {label!r}")
    return int(parts[0]), int(parts[1]), int(parts[2])

def rod_m_from_hk(h: int, k: int) -> int:
    return int(h * h + h * k + k * k)

def build_profile_rod_entries(tilt_deg: int) -> list[dict[str, object]]:
    q_group_by_m = {}
    for row in state.get("geometry", {}).get("q_group_rows", []) or []:
        if not isinstance(row, dict):
            continue
        key = row.get("key")
        try:
            m_value = int(key[2]) if isinstance(key, (list, tuple)) and len(key) >= 3 else None
            qr_value = float(row["qr"])
        except Exception:
            continue
        if m_value is not None and m_value not in q_group_by_m:
            source = str(key[1]) if isinstance(key, (list, tuple)) and len(key) >= 2 else str(row.get("source", "primary"))
            q_group_by_m[m_value] = {"qr": qr_value, "source": source}

    rods = {}
    used_rows = fit_table.loc[fit_table["tilt_deg"] == int(tilt_deg)].copy()
    for _, fit_row in used_rows.iterrows():
        label = str(fit_row["label"])
        try:
            h, k, _l = parse_hkl_label(label)
        except Exception:
            continue
        m_value = rod_m_from_hk(h, k)
        if m_value <= 0 or m_value not in q_group_by_m:
            continue
        q_group = q_group_by_m[m_value]
        rods.setdefault(
            m_value,
            {
                "key": ("q_group", q_group["source"], int(m_value), "rod"),
                "source": q_group["source"],
                "m": int(m_value),
                "qr": float(q_group["qr"]),
            },
        )
    return [rods[m] for m in sorted(rods)]

def projected_qz_at_phi_endpoint(samples: dict[str, object], target_phi: float) -> float | None:
    phi_values = np.asarray(samples.get("phi"), dtype=np.float64)
    qz_values = np.asarray(samples.get("qz"), dtype=np.float64)
    two_theta_values = np.asarray(samples.get("two_theta"), dtype=np.float64)
    if two_theta_values.shape != phi_values.shape:
        two_theta_values = np.full(phi_values.shape, np.nan, dtype=np.float64)
    finite = (
        np.isfinite(phi_values)
        & np.isfinite(qz_values)
        & np.isfinite(two_theta_values)
        & (two_theta_values <= float(ROD_PROFILE_MAX_TWO_THETA_DEG))
    )
    if not np.any(finite):
        return None
    local_index = int(np.nanargmin(np.abs(phi_values[finite] - float(target_phi))))
    absolute_index = np.flatnonzero(finite)[local_index]
    return float(qz_values[absolute_index])

def qz_bounds_for_phi_window(samples: dict[str, object], phi_min: float, phi_max: float) -> tuple[float, float] | None:
    phi_values = np.asarray(samples.get("phi"), dtype=np.float64)
    qz_values = np.asarray(samples.get("qz"), dtype=np.float64)
    two_theta_values = np.asarray(samples.get("two_theta"), dtype=np.float64)
    if two_theta_values.shape != phi_values.shape:
        two_theta_values = np.full(phi_values.shape, np.nan, dtype=np.float64)
    selected = (
        np.isfinite(phi_values)
        & np.isfinite(qz_values)
        & np.isfinite(two_theta_values)
        & (two_theta_values <= float(ROD_PROFILE_MAX_TWO_THETA_DEG))
        & (phi_values >= float(phi_min))
        & (phi_values <= float(phi_max))
    )
    if np.count_nonzero(selected) < 2:
        return None
    branch_edge_phi = float(phi_min) if float(phi_max) <= 0.0 else float(phi_max)
    branch_edge_qz = projected_qz_at_phi_endpoint(samples, branch_edge_phi)
    selected_qz = qz_values[selected]
    if branch_edge_qz is not None and np.isfinite(branch_edge_qz):
        selected_qz = np.concatenate([selected_qz, np.asarray([branch_edge_qz], dtype=np.float64)])
    lo = float(np.nanmin(selected_qz))
    hi = float(np.nanmax(selected_qz))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return None
    return lo, hi

def filter_projected_samples_by_branch_sign(samples: dict[str, object], branch_sign: int | None) -> dict[str, object]:
    if branch_sign is None or "branch_sign" not in samples:
        return samples
    sign_values = np.asarray(samples.get("branch_sign"), dtype=np.int16)
    selected = sign_values == int(branch_sign)
    filtered = dict(samples)
    for key in ("two_theta", "phi", "qz", "branch_sign"):
        values = np.asarray(samples.get(key))
        if values.shape == selected.shape:
            filtered[key] = values[selected]
    filtered["selected_branch_sign"] = int(branch_sign)
    return filtered

def fit_items_for_rod_branch(bg: dict[str, object], rod: dict[str, object], branch_name: str) -> list[dict[str, object]]:
    items = []
    for item in bg["fit_results"]:
        if str(item["branch"]) != str(branch_name):
            continue
        try:
            h, k, _l = parse_hkl_label(str(item["label"]))
        except Exception:
            continue
        if rod_m_from_hk(h, k) == int(rod["m"]):
            items.append(item)
    return items

def select_projected_samples_for_rod_branch(
    bg: dict[str, object],
    rod: dict[str, object],
    samples: dict[str, object],
    *,
    branch_name: str,
    phi_min: float,
    phi_max: float,
) -> tuple[dict[str, object], int | None]:
    sign_values = np.asarray(samples.get("branch_sign", []), dtype=np.int16).reshape(-1)
    if sign_values.size == 0:
        return samples, None
    signs = [int(value) for value in sorted(set(sign_values.tolist())) if int(value) != 0]
    if not signs:
        return samples, None

    tth = np.asarray(samples.get("two_theta"), dtype=np.float64)
    phi = np.asarray(samples.get("phi"), dtype=np.float64)
    qz = np.asarray(samples.get("qz"), dtype=np.float64)
    fit_items = fit_items_for_rod_branch(bg, rod, branch_name)
    scored: list[tuple[float, int]] = []
    for sign in signs:
        selected = (
            (sign_values == sign)
            & np.isfinite(tth)
            & np.isfinite(phi)
            & np.isfinite(qz)
            & (phi >= float(phi_min))
            & (phi <= float(phi_max))
        )
        if not np.any(selected):
            continue
        if fit_items:
            distances = []
            for item in fit_items:
                theta0 = float(item["params"][1])
                phi0 = float(item["params"][2])
                distance = ((tth[selected] - theta0) / 0.6) ** 2 + (wrapped_delta_deg(phi[selected], phi0) / 3.0) ** 2
                distances.append(float(np.nanmin(distance)))
            score = float(np.nanmedian(distances))
        else:
            score = -float(np.count_nonzero(selected))
        scored.append((score, sign))
    if not scored:
        return samples, None
    _score, branch_sign = min(scored, key=lambda item: item[0])
    return filter_projected_samples_by_branch_sign(samples, branch_sign), int(branch_sign)

def config_float_attr(config: object, name: str, default: float = 0.0) -> float:
    try:
        return float(getattr(config, name, default))
    except Exception:
        return float(default)

def notebook_detector_qr_qz_maps(config: object, detector_shape: tuple[int, int]) -> tuple[np.ndarray, np.ndarray, np.ndarray] | None:
    helper = getattr(gui_qr_cylinder_overlay, "detector_qr_qz_maps_for_projection", None)
    if helper is None:
        try:
            reloaded_overlay = importlib.reload(gui_qr_cylinder_overlay)
            globals()["gui_qr_cylinder_overlay"] = reloaded_overlay
            helper = getattr(reloaded_overlay, "detector_qr_qz_maps_for_projection", None)
        except Exception:
            helper = None
    if helper is not None:
        return helper(config=config, detector_shape=detector_shape)

    try:
        height, width = int(detector_shape[0]), int(detector_shape[1])
    except Exception:
        return None
    if height <= 0 or width <= 0:
        return None
    if IntersectionGeometry is None or detector_points_to_sample_qr_qz is None:
        return None
    cols, rows = np.meshgrid(np.arange(width, dtype=np.float64), np.arange(height, dtype=np.float64))
    geometry = IntersectionGeometry(
        image_size=int(config.image_size),
        center_col=float(config.center_col),
        center_row=float(config.center_row),
        distance_cor_to_detector=float(config.distance_cor_to_detector),
        gamma_deg=float(config.gamma_deg),
        Gamma_deg=float(config.Gamma_deg),
        chi_deg=float(config.chi_deg),
        psi_deg=float(config.psi_deg),
        psi_z_deg=float(config.psi_z_deg),
        zs=float(config.zs),
        zb=float(config.zb),
        theta_initial_deg=float(config.theta_initial_deg),
        cor_angle_deg=float(config.cor_angle_deg),
        n_detector=np.array([0.0, 1.0, 0.0], dtype=np.float64),
        unit_x=np.array([1.0, 0.0, 0.0], dtype=np.float64),
        pixel_size_m=float(config.pixel_size_m),
    )
    try:
        return detector_points_to_sample_qr_qz(
            detector_col=cols,
            detector_row=rows,
            geometry=geometry,
            wavelength=float(config.wavelength),
            n2=config.n2,
            beam_x=config_float_attr(config, "beam_x", 0.0),
            beam_y=config_float_attr(config, "beam_y", 0.0),
            dtheta=config_float_attr(config, "dtheta", 0.0),
            dphi=config_float_attr(config, "dphi", 0.0),
        )
    except Exception:
        return None

def caked_qz_map_for_background(
    bg: dict[str, object],
    q_maps: tuple[np.ndarray, np.ndarray, np.ndarray] | None = None,
) -> np.ndarray:
    config = bg["qr_overlay_config"]
    if abs(float(config.theta_initial_deg) - float(ROD_PROFILE_TILT_DEG)) > 1.0e-9:
        raise RuntimeError("caked Qz map must use theta_initial_deg=5")
    if q_maps is None:
        q_maps = notebook_detector_qr_qz_maps(
            config=config,
            detector_shape=tuple(bg["detector_image"].shape),
        )
    if q_maps is None:
        qspace_geometry = exact_qspace_portable.PortableQSpaceGeometry(
            pixel_size_m=PIXEL_SIZE_M,
            distance_m=float(distance_m),
            center_row_px=float(center_row_px),
            center_col_px=float(center_col_px),
            wavelength_m=WAVELENGTH_M,
            gamma_deg=float(config.gamma_deg),
            Gamma_deg=float(config.Gamma_deg),
            chi_deg=float(config.chi_deg),
            psi_deg=float(config.psi_deg),
            psi_z_deg=float(config.psi_z_deg),
            theta_initial_deg=float(config.theta_initial_deg),
            cor_angle_deg=float(config.cor_angle_deg),
            zs=float(config.zs),
            zb=float(config.zb),
        )
        _qx_detector, qz_detector_corners, _qy_detector = exact_qspace_portable._shared_detector_maps_for_shape(  # noqa: SLF001
            tuple(bg["detector_image"].shape),
            qspace_geometry,
        )
        qz_detector_corners = np.asarray(qz_detector_corners, dtype=np.float64)
        qz_detector_center = -0.25 * (
            qz_detector_corners[:-1, :-1]
            + qz_detector_corners[1:, :-1]
            + qz_detector_corners[:-1, 1:]
            + qz_detector_corners[1:, 1:]
        )
    else:
        _qr_detector, qz_detector, valid_q = q_maps
        qz_detector_center = np.asarray(qz_detector, dtype=np.float64).copy()
        qz_detector_center[~np.asarray(valid_q, dtype=bool)] = np.nan
    if qz_detector_center.shape != np.asarray(bg["detector_image"]).shape:
        raise RuntimeError("detector Qz map shape mismatch")
    qz_cake_raw = integrate_detector_to_cake_lut(
        np.asarray(qz_detector_center, dtype=np.float32),
        np.asarray(bg["transform_bundle"].radial_deg, dtype=np.float64),
        np.asarray(bg["transform_bundle"].raw_azimuth_deg, dtype=np.float64),
        bg["transform_bundle"].lut,
    )
    qz_caked, qz_theta_axis, qz_phi_axis = prepare_gui_phi_display(qz_cake_raw)
    if qz_caked.shape != np.asarray(bg["caked_image"]).shape:
        raise RuntimeError("caked Qz map shape mismatch")
    if not np.allclose(qz_theta_axis, np.asarray(bg["theta_axis"], dtype=np.float64)):
        raise RuntimeError("caked Qz theta axis mismatch")
    if not np.allclose(qz_phi_axis, np.asarray(bg["phi_axis"], dtype=np.float64)):
        raise RuntimeError("caked Qz phi axis mismatch")
    return np.asarray(qz_caked, dtype=np.float64)

@njit(parallel=True, nogil=True)
def _profile_accumulate_uniform_numba(
    image: np.ndarray,
    model: np.ndarray,
    mask: np.ndarray,
    qz_map: np.ndarray,
    qz_lo: float,
    qz_hi: float,
    n_bins: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    height, width = image.shape
    row_counts = np.zeros((height, n_bins), dtype=np.int64)
    row_background = np.zeros((height, n_bins), dtype=np.float64)
    row_fit = np.zeros((height, n_bins), dtype=np.float64)
    if n_bins <= 0 or qz_hi <= qz_lo:
        return np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.float64), np.zeros(0, dtype=np.float64)
    scale = float(n_bins) / (qz_hi - qz_lo)
    for row in prange(height):
        for col in range(width):
            if not mask[row, col]:
                continue
            qz = qz_map[row, col]
            background_value = image[row, col]
            fit_value = model[row, col]
            if not (np.isfinite(qz) and np.isfinite(background_value) and np.isfinite(fit_value)):
                continue
            if qz < qz_lo or qz >= qz_hi:
                continue
            bin_index = int((qz - qz_lo) * scale)
            if 0 <= bin_index < n_bins:
                row_counts[row, bin_index] += 1
                row_background[row, bin_index] += background_value
                row_fit[row, bin_index] += fit_value
    counts = np.zeros(n_bins, dtype=np.int64)
    background_sum = np.zeros(n_bins, dtype=np.float64)
    fit_sum = np.zeros(n_bins, dtype=np.float64)
    for row in range(height):
        for bin_index in range(n_bins):
            counts[bin_index] += row_counts[row, bin_index]
            background_sum[bin_index] += row_background[row, bin_index]
            fit_sum[bin_index] += row_fit[row, bin_index]
    return counts, background_sum, fit_sum


@njit(nogil=True)
def _find_marker_peak_in_window_numba(
    theta_axis: np.ndarray,
    phi_axis: np.ndarray,
    model: np.ndarray,
    image: np.ndarray,
    branch_mask: np.ndarray,
    qz_map: np.ndarray,
    theta0: float,
    phi0: float,
    theta_half_width: float,
    phi_half_width: float,
) -> tuple[int, int]:
    best_model = -np.inf
    best_model_row = -1
    best_model_col = -1
    best_image = -np.inf
    best_image_row = -1
    best_image_col = -1
    any_local = False
    for row in range(phi_axis.size):
        phi = phi_axis[row]
        if abs(_wrapped_delta_deg_scalar_numba(phi, phi0)) > phi_half_width:
            continue
        for col in range(theta_axis.size):
            theta = theta_axis[col]
            if abs(theta - theta0) > theta_half_width:
                continue
            if not branch_mask[row, col]:
                continue
            if not np.isfinite(qz_map[row, col]):
                continue
            any_local = True
            model_value = model[row, col]
            if np.isfinite(model_value) and model_value > 0.0 and model_value > best_model:
                best_model = model_value
                best_model_row = row
                best_model_col = col
            image_value = image[row, col]
            if np.isfinite(image_value) and image_value > best_image:
                best_image = image_value
                best_image_row = row
                best_image_col = col
    if best_model_row >= 0:
        return best_model_row, best_model_col
    if any_local and best_image_row >= 0:
        return best_image_row, best_image_col
    return -1, -1


@njit(nogil=True)
def _refine_marker_to_local_peak_numba(
    theta_axis: np.ndarray,
    phi_axis: np.ndarray,
    model: np.ndarray,
    image: np.ndarray,
    branch_mask: np.ndarray,
    qz_map: np.ndarray,
    theta0: float,
    phi0: float,
    theta_half_width: float,
    phi_half_width: float,
    fallback_theta_half_width: float,
    fallback_phi_half_width: float,
) -> tuple[int, float, float, float]:
    row, col = _find_marker_peak_in_window_numba(theta_axis, phi_axis, model, image, branch_mask, qz_map, theta0, phi0, theta_half_width, phi_half_width)
    if row < 0:
        row, col = _find_marker_peak_in_window_numba(theta_axis, phi_axis, model, image, branch_mask, qz_map, theta0, phi0, fallback_theta_half_width, fallback_phi_half_width)
    if row < 0:
        return 0, np.nan, np.nan, np.nan
    return 1, qz_map[row, col], theta_axis[col], phi_axis[row]


def profile_from_full_mask(
    bg: dict[str, object],
    rod: dict[str, object],
    *,
    branch_name: str,
    branch_label: str,
    qz_edges: np.ndarray,
    mask: np.ndarray,
    qz_map: np.ndarray,
) -> pd.DataFrame:
    image = np.asarray(bg["caked_image"], dtype=np.float64)
    theta_axis = np.asarray(bg["theta_axis"], dtype=np.float64).reshape(-1)
    if image.ndim != 2 or theta_axis.size != image.shape[1]:
        raise RuntimeError("caked image/theta axis shape mismatch")
    theta_map = np.broadcast_to(theta_axis[None, :], image.shape)
    payload = qz_profile_from_caked_mask(
        image=image,
        model=np.asarray(bg["caked_peak_model"], dtype=np.float64),
        qz_map=np.asarray(qz_map, dtype=np.float64),
        qz_edges=np.asarray(qz_edges, dtype=np.float64),
        mask=np.asarray(mask, dtype=bool),
        signal_sum=bg.get("caked_sum_signal"),
        normalization_sum=bg.get("caked_sum_normalization"),
        theta_map=theta_map,
    )
    rows = pd.DataFrame(payload)
    metadata = {
        "sample_name": SAMPLE_NAME,
        "tilt_deg": int(bg["tilt_deg"]),
        "theta_initial_deg_used_for_q": float(bg["qr_overlay_config"].theta_initial_deg),
        "qz_map_source": "detector_qspace_reprojected_to_caked",
        "qr_integration_source": "caked_selected_qr_mask",
        "solid_angle_corrected": True,
        "m": int(rod["m"]),
        "qr": float(rod["qr"]),
        "branch": str(branch_name),
        "phi_window": str(branch_label),
        "delta_qr": float(qr_rod_delta_qr),
    }
    for key, value in reversed(list(metadata.items())):
        rows.insert(0, key, value)
    return rows

def detector_gui_phi_map_for_background(bg: dict[str, object]) -> np.ndarray | None:
    config = bg["qr_overlay_config"]
    detector_shape = tuple(bg["detector_image"].shape)
    helper = getattr(gui_qr_cylinder_overlay, "detector_gui_phi_map_for_projection", None)
    if helper is not None:
        try:
            phi = helper(config=config, detector_shape=detector_shape)
            if phi is not None:
                return np.asarray(phi, dtype=np.float64)
        except Exception:
            pass
    try:
        ai = FastAzimuthalIntegrator(
            dist=float(distance_m),
            poni1=float(center_row_px) * PIXEL_SIZE_M,
            poni2=float(center_col_px) * PIXEL_SIZE_M,
            pixel1=PIXEL_SIZE_M,
            pixel2=PIXEL_SIZE_M,
            wavelength=WAVELENGTH_M,
        )
        _theta_map, raw_phi_map = detector_pixel_angular_maps(detector_shape, ai.geometry)
        return np.asarray(raw_phi_to_gui_phi(raw_phi_map), dtype=np.float64)
    except Exception:
        return None


def detector_solid_angle_for_background(bg: dict[str, object]) -> np.ndarray | None:
    config = bg["qr_overlay_config"]
    detector_shape = tuple(bg["detector_image"].shape)
    helper = getattr(gui_qr_cylinder_overlay, "detector_solid_angle_for_projection", None)
    if helper is not None:
        try:
            solid_angle = helper(config=config, detector_shape=detector_shape)
            if solid_angle is not None:
                return np.asarray(solid_angle, dtype=np.float64)
        except Exception:
            pass
    try:
        ai = FastAzimuthalIntegrator(
            dist=float(distance_m),
            poni1=float(center_row_px) * PIXEL_SIZE_M,
            poni2=float(center_col_px) * PIXEL_SIZE_M,
            pixel1=PIXEL_SIZE_M,
            pixel2=PIXEL_SIZE_M,
            wavelength=WAVELENGTH_M,
        )
        if hasattr(ai, "_solid_angle_for_shape"):
            return np.asarray(ai._solid_angle_for_shape(detector_shape), dtype=np.float64)  # noqa: SLF001
    except Exception:
        pass
    return None


def detector_two_theta_map_for_background(bg: dict[str, object]) -> np.ndarray | None:
    detector_shape = tuple(bg["detector_image"].shape)
    try:
        ai = FastAzimuthalIntegrator(
            dist=float(distance_m),
            poni1=float(center_row_px) * PIXEL_SIZE_M,
            poni2=float(center_col_px) * PIXEL_SIZE_M,
            pixel1=PIXEL_SIZE_M,
            pixel2=PIXEL_SIZE_M,
            wavelength=WAVELENGTH_M,
        )
        theta_map, _raw_phi_map = detector_pixel_angular_maps(detector_shape, ai.geometry)
        return np.asarray(theta_map, dtype=np.float64)
    except Exception:
        return None


def profile_from_detector_qr_qz(
    bg: dict[str, object],
    rod: dict[str, object],
    *,
    branch_name: str,
    branch_label: str,
    qz_edges: np.ndarray,
    phi_min: float,
    phi_max: float,
    detector_q_maps: tuple[np.ndarray, np.ndarray, np.ndarray],
    detector_phi_map: np.ndarray,
    detector_solid_angle: np.ndarray | None,
    detector_two_theta_map: np.ndarray | None,
) -> pd.DataFrame:
    image = np.asarray(bg["detector_image"], dtype=np.float64)
    model = np.asarray(bg["detector_peak_model"], dtype=np.float64)
    qr_map = np.asarray(detector_q_maps[0], dtype=np.float64)
    qz_map = np.asarray(detector_q_maps[1], dtype=np.float64)
    valid_q = np.asarray(detector_q_maps[2], dtype=bool)
    phi_map = np.asarray(detector_phi_map, dtype=np.float64)
    edges = np.asarray(qz_edges, dtype=np.float64).reshape(-1)
    if (
        image.ndim != 2
        or model.shape != image.shape
        or qr_map.shape != image.shape
        or qz_map.shape != image.shape
        or valid_q.shape != image.shape
        or phi_map.shape != image.shape
        or edges.size < 2
        or not np.all(np.isfinite(edges))
        or not np.all(np.diff(edges) > 0.0)
    ):
        return pd.DataFrame()

    if detector_solid_angle is None:
        signal = image
        finite_solid = np.ones(image.shape, dtype=bool)
        solid_angle_corrected = False
    else:
        solid_angle = np.asarray(detector_solid_angle, dtype=np.float64)
        if solid_angle.shape != image.shape:
            return pd.DataFrame()
        finite_solid = np.isfinite(solid_angle) & (solid_angle > 0.0)
        signal = np.full(image.shape, np.nan, dtype=np.float64)
        signal[finite_solid] = image[finite_solid] / solid_angle[finite_solid]
        solid_angle_corrected = True

    theta_map = None
    if detector_two_theta_map is None:
        below_theta_limit = np.ones(image.shape, dtype=bool)
    else:
        theta_map = np.asarray(detector_two_theta_map, dtype=np.float64)
        if theta_map.shape != image.shape:
            return pd.DataFrame()
        below_theta_limit = np.isfinite(theta_map) & (theta_map <= float(ROD_PROFILE_MAX_TWO_THETA_DEG))

    qr0 = float(rod["qr"])
    delta_qr = float(qr_rod_delta_qr)
    qr_lo = max(0.0, qr0 - delta_qr)
    qr_hi = qr0 + delta_qr
    if float(phi_min) <= float(phi_max):
        phi_selected = (phi_map >= float(phi_min)) & (phi_map <= float(phi_max))
    else:
        phi_selected = (phi_map >= float(phi_min)) | (phi_map <= float(phi_max))
    base = (
        valid_q
        & finite_solid
        & below_theta_limit
        & phi_selected
        & np.isfinite(qr_map)
        & np.isfinite(qz_map)
        & (qr_map >= qr_lo)
        & (qr_map <= qr_hi)
    )
    selected_qz = qz_map[base]
    if selected_qz.size == 0:
        return pd.DataFrame()
    background_values = np.where(np.isfinite(signal[base]), signal[base], 0.0)
    fit_values = np.where(np.isfinite(model[base]), model[base], 0.0)
    counts, _ = np.histogram(selected_qz, bins=edges)
    background_sums, _ = np.histogram(selected_qz, bins=edges, weights=background_values)
    fit_sums, _ = np.histogram(selected_qz, bins=edges, weights=fit_values)
    counts = np.asarray(counts, dtype=np.int64)
    background_sums = np.asarray(background_sums, dtype=np.float64)
    fit_sums = np.asarray(fit_sums, dtype=np.float64)
    background_mean = np.full(counts.shape, np.nan, dtype=np.float64)
    fit_mean = np.full(counts.shape, np.nan, dtype=np.float64)
    nonempty = counts > 0
    background_mean[nonempty] = background_sums[nonempty] / counts[nonempty].astype(np.float64)
    fit_mean[nonempty] = fit_sums[nonempty] / counts[nonempty].astype(np.float64)

    two_theta_min = np.full(counts.shape, np.nan, dtype=np.float64)
    two_theta_max = np.full(counts.shape, np.nan, dtype=np.float64)
    two_theta_mean = np.full(counts.shape, np.nan, dtype=np.float64)
    if theta_map is not None:
        theta_values = theta_map[base]
        for bin_index, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
            if bin_index == counts.size - 1:
                in_bin = (selected_qz >= lo) & (selected_qz <= hi)
            else:
                in_bin = (selected_qz >= lo) & (selected_qz < hi)
            finite_theta = theta_values[in_bin & np.isfinite(theta_values)]
            if finite_theta.size:
                two_theta_min[bin_index] = float(np.nanmin(finite_theta))
                two_theta_max[bin_index] = float(np.nanmax(finite_theta))
                two_theta_mean[bin_index] = float(np.nanmean(finite_theta))

    return pd.DataFrame({
        "sample_name": [SAMPLE_NAME] * int(counts.size),
        "tilt_deg": [int(bg["tilt_deg"])] * int(counts.size),
        "theta_initial_deg_used_for_q": [float(bg["qr_overlay_config"].theta_initial_deg)] * int(counts.size),
        "qz_map_source": ["detector_qspace_per_pixel"] * int(counts.size),
        "qr_integration_source": ["detector_qr_band_per_qz_bin"] * int(counts.size),
        "solid_angle_corrected": [solid_angle_corrected] * int(counts.size),
        "m": [int(rod["m"])] * int(counts.size),
        "qr": [qr0] * int(counts.size),
        "branch": [str(branch_name)] * int(counts.size),
        "phi_window": [str(branch_label)] * int(counts.size),
        "delta_qr": [delta_qr] * int(counts.size),
        "qz_bin": np.arange(1, int(counts.size) + 1, dtype=int),
        "qz_min": edges[:-1],
        "qz_max": edges[1:],
        "qz_center": 0.5 * (edges[:-1] + edges[1:]),
        "pixel_count": counts,
        "acceptance_sum": counts.astype(np.float64),
        "acceptance_source": ["pixel_count"] * int(counts.size),
        "background_sum": np.where(nonempty, background_sums, np.nan),
        "fit_sum": np.where(nonempty, fit_sums, np.nan),
        "background_mean": background_mean,
        "fit_mean": fit_mean,
        "background_weighted_sum": np.where(nonempty, background_sums, np.nan),
        "fit_weighted_sum": np.where(nonempty, fit_sums, np.nan),
        "background_density": background_mean,
        "fit_density": fit_mean,
        "two_theta_min": two_theta_min,
        "two_theta_max": two_theta_max,
        "two_theta_mean": two_theta_mean,
    })

def rolling_lower_envelope(values: object, *, fraction: float = 0.22, quantile: float = 18.0) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float64)
    baseline = np.full(arr.shape, np.nan, dtype=np.float64)
    finite = np.isfinite(arr)
    if np.count_nonzero(finite) < 3:
        fill = float(np.nanpercentile(arr[finite], quantile)) if np.any(finite) else 0.0
        return np.full(arr.shape, fill, dtype=np.float64)
    finite_values = arr[finite]
    fallback = float(np.nanpercentile(finite_values, quantile))
    n = int(arr.size)
    window = max(7, int(round(n * float(fraction))))
    if window % 2 == 0:
        window += 1
    half = window // 2
    for idx in range(n):
        lo = max(0, idx - half)
        hi = min(n, idx + half + 1)
        local = arr[lo:hi]
        local = local[np.isfinite(local)]
        baseline[idx] = float(np.nanpercentile(local, quantile)) if local.size else fallback
    smooth = max(5, window // 3)
    if smooth % 2 == 0:
        smooth += 1
    pad = smooth // 2
    padded = np.pad(np.where(np.isfinite(baseline), baseline, fallback), (pad, pad), mode="edge")
    kernel = np.ones(smooth, dtype=np.float64) / float(smooth)
    return np.convolve(padded, kernel, mode="valid")

def normalized_profile_pair(background_density: pd.Series, fit_density: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    measured = np.asarray(background_density, dtype=np.float64)
    fitted = np.asarray(fit_density, dtype=np.float64)
    baseline = rolling_lower_envelope(measured)
    measured_net = measured - baseline
    positive_values = np.concatenate([
        measured_net[np.isfinite(measured_net) & (measured_net > 0.0)],
        fitted[np.isfinite(fitted) & (fitted > 0.0)],
    ])
    scale = float(np.nanpercentile(positive_values, 99.0)) if positive_values.size else 1.0
    if not np.isfinite(scale) or scale <= 0.0:
        scale = 1.0
    return measured_net / scale, fitted / scale

def gaussian_sum_qz_model(x_values: object, amplitudes: object, centers: object, sigmas: object) -> np.ndarray:
    x = np.asarray(x_values, dtype=np.float64)
    amps = np.asarray(amplitudes, dtype=np.float64).reshape(-1)
    ctrs = np.asarray(centers, dtype=np.float64).reshape(-1)
    widths = np.maximum(np.asarray(sigmas, dtype=np.float64).reshape(-1), 1.0e-12)
    model = np.zeros(x.shape, dtype=np.float64)
    for amp, center, sigma in zip(amps, ctrs, widths, strict=False):
        model += float(amp) * np.exp(-0.5 * ((x - float(center)) / float(sigma)) ** 2)
    return model

def _unique_sorted_markers(values: object, *, min_separation: float) -> np.ndarray:
    markers = np.asarray(values, dtype=np.float64).reshape(-1)
    markers = np.sort(markers[np.isfinite(markers)])
    if markers.size <= 1:
        return markers
    kept = [float(markers[0])]
    for value in markers[1:]:
        if abs(float(value) - kept[-1]) >= float(min_separation):
            kept.append(float(value))
    return np.asarray(kept, dtype=np.float64)

def fit_joint_qz_peak_sum(qz_center: object, background_density: object, qz_markers: object) -> dict[str, object]:
    x = np.asarray(qz_center, dtype=np.float64).reshape(-1)
    measured = np.asarray(background_density, dtype=np.float64).reshape(-1)
    if x.size != measured.size or x.size < 6:
        return {"success": False, "message": "bad_profile_shape", "model_density": np.full(x.shape, np.nan)}
    baseline = rolling_lower_envelope(measured)
    y = measured - baseline
    finite = np.isfinite(x) & np.isfinite(y)
    if np.count_nonzero(finite) < 6:
        return {"success": False, "message": "too_few_profile_points", "model_density": np.full(x.shape, np.nan)}
    x_fit = np.asarray(x[finite], dtype=np.float64)
    y_fit = np.asarray(y[finite], dtype=np.float64)
    order = np.argsort(x_fit)
    x_fit = x_fit[order]
    y_fit = y_fit[order]
    diffs = np.diff(np.unique(x_fit))
    dx = float(np.nanmedian(diffs)) if diffs.size else 1.0e-3
    dx = max(dx, 1.0e-6)
    x_min = float(np.nanmin(x_fit))
    x_max = float(np.nanmax(x_fit))
    x_span = max(x_max - x_min, dx * 10.0)
    markers = _unique_sorted_markers(qz_markers, min_separation=0.5 * dx)
    markers = markers[(markers >= x_min) & (markers <= x_max)]
    if markers.size < 1:
        return {"success": False, "message": "no_qz_markers", "model_density": np.full(x.shape, np.nan)}
    n_peaks = int(markers.size)
    if np.count_nonzero(finite) < max(6, 2 * n_peaks + 2):
        return {"success": False, "message": "too_few_points_for_markers", "model_density": np.full(x.shape, np.nan)}
    if n_peaks == 1:
        nearest_spacing = np.full(1, x_span, dtype=np.float64)
    else:
        nearest_spacing = np.empty(n_peaks, dtype=np.float64)
        for idx, center in enumerate(markers):
            distances = np.abs(markers - float(center))
            distances = distances[distances > 0.0]
            nearest_spacing[idx] = float(np.nanmin(distances)) if distances.size else x_span
    center_half_width = np.maximum(2.0 * dx, np.minimum(0.12 * x_span, 0.35 * nearest_spacing))
    center_lower = np.maximum(x_min, markers - center_half_width)
    center_upper = np.minimum(x_max, markers + center_half_width)
    min_sigma = max(0.45 * dx, x_span / 900.0)
    max_sigma = np.maximum(1.6 * min_sigma, np.minimum(0.08 * x_span, 0.42 * nearest_spacing))
    sigma0 = np.minimum(max_sigma, np.maximum(1.4 * dx, 0.16 * nearest_spacing))
    y_positive = y_fit[np.isfinite(y_fit) & (y_fit > 0.0)]
    peak_scale = float(np.nanpercentile(y_positive, 98.0)) if y_positive.size else max(float(np.nanmax(y_fit)), 1.0)
    peak_scale = max(peak_scale, 1.0e-9)
    amp0 = np.maximum(np.interp(markers, x_fit, np.clip(y_fit, 0.0, None), left=0.0, right=0.0), 0.08 * peak_scale)
    amp_upper = np.full(n_peaks, max(30.0 * peak_scale, peak_scale + 1.0), dtype=np.float64)
    p0 = np.concatenate([amp0, markers, sigma0])
    lower = np.concatenate([np.zeros(n_peaks, dtype=np.float64), center_lower, np.full(n_peaks, min_sigma, dtype=np.float64)])
    upper = np.concatenate([amp_upper, center_upper, max_sigma])
    p0 = np.minimum(np.maximum(p0, lower + 1.0e-12), upper - 1.0e-12)
    residual_scale = max(float(np.nanpercentile(np.abs(y_fit), 90.0)), 1.0e-9)

    def residual(params: np.ndarray) -> np.ndarray:
        amps = params[:n_peaks]
        centers = params[n_peaks : 2 * n_peaks]
        sigmas = params[2 * n_peaks : 3 * n_peaks]
        model = gaussian_sum_qz_model(x_fit, amps, centers, sigmas)
        res = (model - y_fit) / residual_scale
        nearest_scaled = np.full(x_fit.shape, np.inf, dtype=np.float64)
        for center, sigma in zip(centers, sigmas, strict=False):
            nearest_scaled = np.minimum(nearest_scaled, np.abs((x_fit - float(center)) / max(float(sigma), min_sigma)))
        over = res > 0.0
        res[over] *= 1.0 + 1.6 * np.clip((nearest_scaled[over] - 0.75) / 2.0, 0.0, 1.0)
        return res

    result = least_squares(
        residual,
        p0,
        bounds=(lower, upper),
        loss="soft_l1",
        f_scale=1.0,
        max_nfev=2000,
    )
    params = np.asarray(result.x, dtype=np.float64)
    model_density = gaussian_sum_qz_model(x, params[:n_peaks], params[n_peaks : 2 * n_peaks], params[2 * n_peaks : 3 * n_peaks])
    return {
        "success": bool(result.success),
        "message": str(result.message),
        "model_density": model_density,
        "peak_count": n_peaks,
        "params": params,
        "cost": float(result.cost),
    }

def marker_qz_values_for_profile(marker_table: pd.DataFrame, *, m_value: int, branch_value: str) -> np.ndarray:
    if marker_table.empty or "qz_marker" not in marker_table:
        return np.array([], dtype=np.float64)
    sub = marker_table[
        (np.asarray(marker_table["m"], dtype=int) == int(m_value))
        & (marker_table["branch"].astype(str) == str(branch_value))
    ]
    return np.asarray(sub["qz_marker"], dtype=np.float64)

def add_joint_qz_fit_columns(rod_profile_table: pd.DataFrame, marker_table: pd.DataFrame) -> pd.DataFrame:
    table = rod_profile_table.copy()
    table["joint_fit_density"] = np.asarray(table.get("fit_density", np.nan), dtype=np.float64)
    table["joint_fit_success"] = False
    table["joint_fit_peak_count"] = 0
    table["joint_fit_message"] = "fallback_detector_peak_model"
    if table.empty:
        return table
    for (m_value, branch_value), sub in table.groupby(["m", "branch"], sort=False):
        qz_markers = marker_qz_values_for_profile(marker_table, m_value=int(m_value), branch_value=str(branch_value))
        fit_payload = fit_joint_qz_peak_sum(sub["qz_center"], sub["background_density"], qz_markers)
        if not bool(fit_payload.get("success")):
            table.loc[sub.index, "joint_fit_message"] = str(fit_payload.get("message", "joint_fit_failed"))
            continue
        table.loc[sub.index, "joint_fit_density"] = np.asarray(fit_payload["model_density"], dtype=np.float64)
        table.loc[sub.index, "joint_fit_success"] = True
        table.loc[sub.index, "joint_fit_peak_count"] = int(fit_payload.get("peak_count", 0))
        table.loc[sub.index, "joint_fit_message"] = str(fit_payload.get("message", ""))
    return table

def nearest_projected_qz_for_fit(item: dict[str, object], samples: dict[str, object], phi_min: float, phi_max: float) -> tuple[float, float, float] | None:
    tth = np.asarray(samples.get("two_theta"), dtype=np.float64)
    phi = np.asarray(samples.get("phi"), dtype=np.float64)
    qz = np.asarray(samples.get("qz"), dtype=np.float64)
    theta0 = float(item["params"][1])
    phi0 = float(item["params"][2])
    selected = (
        np.isfinite(tth)
        & np.isfinite(phi)
        & np.isfinite(qz)
        & (tth <= float(ROD_PROFILE_MAX_TWO_THETA_DEG))
        & (phi > float(phi_min))
        & (phi < float(phi_max))
    )
    if not np.any(selected):
        return None
    distance = ((tth[selected] - theta0) / 0.5) ** 2 + (wrapped_delta_deg(phi[selected], phi0) / 2.0) ** 2
    local_idx = int(np.nanargmin(distance))
    absolute_idx = np.flatnonzero(selected)[local_idx]
    return float(qz[absolute_idx]), float(tth[absolute_idx]), float(phi[absolute_idx])

def refine_marker_to_local_peak(bg: dict[str, object], item: dict[str, object], branch_mask: np.ndarray, qz_map: np.ndarray) -> tuple[float, float, float] | None:
    theta_axis = np.ascontiguousarray(bg["theta_axis"], dtype=np.float64)
    phi_axis = np.ascontiguousarray(bg["phi_axis"], dtype=np.float64)
    model = np.ascontiguousarray(bg["caked_peak_model"], dtype=np.float64)
    image = np.ascontiguousarray(bg["caked_image"], dtype=np.float64)
    params = np.asarray(item["params"], dtype=np.float64)
    theta0 = float(params[1])
    phi0 = float(params[2])
    theta_half_width = max(0.35, 3.0 * float(abs(params[3])))
    phi_half_width = max(1.2, 3.0 * float(abs(params[4])))
    success, qz_value, theta_value, phi_value = _refine_marker_to_local_peak_numba(
        theta_axis,
        phi_axis,
        model,
        image,
        np.ascontiguousarray(branch_mask, dtype=np.bool_),
        np.ascontiguousarray(qz_map, dtype=np.float64),
        theta0,
        phi0,
        theta_half_width,
        phi_half_width,
        THETA_HALF_WINDOW_DEG,
        PHI_HALF_WINDOW_DEG,
    )
    if not success:
        return None
    return float(qz_value), float(theta_value), float(phi_value)

def marker_rows_for_rod_branch(
    bg: dict[str, object],
    rod: dict[str, object],
    samples: dict[str, object],
    *,
    branch_name: str,
    branch_label: str,
    phi_min: float,
    phi_max: float,
    branch_mask: np.ndarray,
    qz_map: np.ndarray,
) -> list[dict[str, object]]:
    rows = []
    for item in bg["fit_results"]:
        if str(item["branch"]) != str(branch_name):
            continue
        try:
            h, k, l_value = parse_hkl_label(str(item["label"]))
        except Exception:
            continue
        if rod_m_from_hk(h, k) != int(rod["m"]):
            continue
        nearest = nearest_projected_qz_for_fit(item, samples, phi_min, phi_max)
        if nearest is None:
            continue
        qz_value, projected_tth, projected_phi = nearest
        refined = refine_marker_to_local_peak(bg, item, branch_mask, qz_map)
        if refined is None:
            refined_qz, refined_tth, refined_phi = qz_value, projected_tth, projected_phi
            marker_source = "projected_nearest"
        else:
            refined_qz, refined_tth, refined_phi = refined
            marker_source = "local_peak_model_max"
        if (
            not np.isfinite(refined_tth)
            or float(refined_tth) > float(ROD_PROFILE_MAX_TWO_THETA_DEG)
        ):
            continue
        rows.append({
            "sample_name": SAMPLE_NAME,
            "tilt_deg": int(bg["tilt_deg"]),
            "theta_initial_deg_used_for_q": float(bg["qr_overlay_config"].theta_initial_deg),
            "m": int(rod["m"]),
            "qr": float(rod["qr"]),
            "branch": str(branch_name),
            "phi_window": str(branch_label),
            "hkl": str(item["label"]),
            "l": int(l_value),
            "fit_two_theta_deg": float(item["params"][1]),
            "fit_phi_deg": float(item["params"][2]),
            "projected_two_theta_deg": projected_tth,
            "projected_phi_deg": projected_phi,
            "projected_qz_marker": qz_value,
            "refined_two_theta_deg": refined_tth,
            "refined_phi_deg": refined_phi,
            "qz_marker": refined_qz,
            "marker_source": marker_source,
        })
    return sorted(rows, key=lambda row: row["qz_marker"])

profile_bg = next(bg for bg in ordered_backgrounds if int(bg["tilt_deg"]) == ROD_PROFILE_TILT_DEG)
if abs(float(profile_bg["qr_overlay_config"].theta_initial_deg) - float(ROD_PROFILE_TILT_DEG)) > 1.0e-9:
    raise RuntimeError("Qr profile config is not using theta_initial_deg=5 for Q conversion")
profile_detector_q_maps = notebook_detector_qr_qz_maps(
    config=profile_bg["qr_overlay_config"],
    detector_shape=tuple(profile_bg["detector_image"].shape),
)
if profile_detector_q_maps is None:
    raise RuntimeError("detector Qr/Qz maps are required for detector-space Qr rod profiles")
profile_detector_phi_map = detector_gui_phi_map_for_background(profile_bg)
if profile_detector_phi_map is None:
    raise RuntimeError("detector phi map is required for detector-space Qr rod profiles")
profile_detector_solid_angle = detector_solid_angle_for_background(profile_bg)
profile_detector_two_theta_map = detector_two_theta_map_for_background(profile_bg)
if profile_detector_two_theta_map is None:
    raise RuntimeError("detector 2theta map is required for 60 deg Qr rod profile cutoff")
profile_qz_map = caked_qz_map_for_background(profile_bg, q_maps=profile_detector_q_maps)
theta_region_axis = np.asarray(profile_bg["theta_axis"], dtype=np.float64)
phi_region_axis = np.asarray(profile_bg["phi_axis"], dtype=np.float64)
theta_region_within_profile_limit = (
    np.isfinite(theta_region_axis)
    & (theta_region_axis <= float(ROD_PROFILE_MAX_TWO_THETA_DEG))
)
specular_region_mask = (
    (theta_region_axis[None, :] >= 5.0)
    & (theta_region_axis[None, :] <= 25.0)
    & (phi_region_axis[:, None] > -10.0)
    & (phi_region_axis[:, None] < 10.0)
)

rod_entries = build_profile_rod_entries(ROD_PROFILE_TILT_DEG)
if not rod_entries:
    raise RuntimeError("no non-specular Qr rods were available for 5 deg branch plotting")
phi_windows = [
    ("-", -90.0, 0.0, r"$-90^\circ<\phi<0^\circ$"),
    ("+", 0.0, 90.0, r"$0^\circ<\phi<90^\circ$"),
]
profile_rows = []
marker_rows = []
region_overlays = []
profile_failures = []
specular_rod_entry = {"key": ("q_group", "primary", 0, "qz_rod"), "source": "primary", "m": 0, "qr": 0.0}
specular_qz_values = profile_qz_map[specular_region_mask & np.isfinite(profile_qz_map)]
if specular_qz_values.size >= 2:
    specular_edges = np.linspace(float(np.nanmin(specular_qz_values)), float(np.nanmax(specular_qz_values)), ROD_PROFILE_QZ_BINS + 1, dtype=np.float64)
    profile_rows.append(
        profile_from_full_mask(
            profile_bg,
            specular_rod_entry,
            branch_name="qz",
            branch_label=r"$-10^\circ<\phi<10^\circ$, $5^\circ<2\theta<25^\circ$",
            qz_edges=specular_edges,
            mask=specular_region_mask,
            qz_map=profile_qz_map,
        )
    )
else:
    profile_failures.append((ROD_PROFILE_TILT_DEG, 0.0, "H=K=0", "no caked qz values"))

for rod in rod_entries:
    samples = gui_qr_cylinder_overlay.project_selected_qr_rod_caked_samples(
        selected_entry=rod,
        config=profile_bg["qr_overlay_config"],
        projection_context=profile_bg["caked_projection_context"],
    )
    if samples is None:
        profile_failures.append((ROD_PROFILE_TILT_DEG, float(rod["qr"]), "projected samples unavailable"))
        continue
    for branch_name, phi_min, phi_max, branch_label in phi_windows:
        branch_samples, branch_sign = select_projected_samples_for_rod_branch(
            profile_bg,
            rod,
            samples,
            branch_name=branch_name,
            phi_min=phi_min,
            phi_max=phi_max,
        )
        qz_bounds = qz_bounds_for_phi_window(branch_samples, phi_min, phi_max)
        if qz_bounds is None:
            profile_failures.append((ROD_PROFILE_TILT_DEG, float(rod["qr"]), branch_label, "no projected qz span"))
            continue
        qz_lo, qz_hi = qz_bounds
        mask_payload = gui_qr_cylinder_overlay.build_selected_qr_rod_qz_caked_mask(
            selected_entry=rod,
            config=profile_bg["qr_overlay_config"],
            projection_context=profile_bg["caked_projection_context"],
            radial_axis=np.asarray(profile_bg["theta_axis"], dtype=np.float64),
            azimuth_axis=np.asarray(profile_bg["phi_axis"], dtype=np.float64),
            delta_qr=qr_rod_delta_qr,
            qz_min=qz_lo,
            qz_max=qz_hi,
            phi_min=phi_min,
            phi_max=phi_max,
        )
        if not isinstance(mask_payload, dict):
            profile_failures.append((ROD_PROFILE_TILT_DEG, float(rod["qr"]), branch_label, "mask unavailable"))
            continue
        branch_mask = np.asarray(mask_payload.get("mask"), dtype=bool)
        if branch_mask.shape != np.asarray(profile_bg["caked_image"]).shape:
            profile_failures.append((ROD_PROFILE_TILT_DEG, float(rod["qr"]), branch_label, "mask shape mismatch"))
            continue
        branch_mask = branch_mask & theta_region_within_profile_limit[None, :]
        if not np.any(branch_mask):
            profile_failures.append((ROD_PROFILE_TILT_DEG, float(rod["qr"]), branch_label, "empty mask"))
            continue
        qz_edge_lo, qz_edge_hi = sorted((float(qz_lo), float(qz_hi)))
        edges = np.linspace(qz_edge_lo, qz_edge_hi, ROD_PROFILE_QZ_BINS + 1, dtype=np.float64)
        detector_profile = profile_from_detector_qr_qz(
            profile_bg,
            rod,
            branch_name=branch_name,
            branch_label=branch_label,
            qz_edges=edges,
            phi_min=phi_min,
            phi_max=phi_max,
            detector_q_maps=profile_detector_q_maps,
            detector_phi_map=profile_detector_phi_map,
            detector_solid_angle=profile_detector_solid_angle,
            detector_two_theta_map=profile_detector_two_theta_map,
        )
        if detector_profile.empty or int(np.nansum(detector_profile["pixel_count"])) <= 0:
            profile_failures.append((ROD_PROFILE_TILT_DEG, float(rod["qr"]), branch_label, "empty detector Qr/Qz profile"))
            continue
        region_overlays.append({
            "m": int(rod["m"]),
            "qr": float(rod["qr"]),
            "branch": str(branch_name),
            "branch_sign": None if branch_sign is None else int(branch_sign),
            "branch_label": str(branch_label),
            "qz_min": qz_edge_lo,
            "qz_max": qz_edge_hi,
            "mask": branch_mask.copy(),
        })
        profile_rows.append(detector_profile)
        marker_rows.extend(
            marker_rows_for_rod_branch(
                profile_bg,
                rod,
                branch_samples,
                branch_name=branch_name,
                branch_label=branch_label,
                phi_min=phi_min,
                phi_max=phi_max,
                branch_mask=branch_mask,
                qz_map=profile_qz_map,
            )
        )

if not profile_rows:
    raise RuntimeError("no 5 deg Qr rod profiles were generated")

marker_table = pd.DataFrame(marker_rows)
rod_profile_table = pd.concat(profile_rows, ignore_index=True)
rod_profile_table = add_joint_qz_fit_columns(rod_profile_table, marker_table)
rod_profile_csv = OUT_DIR / f"{ROD_PROFILE_STEM}.csv"
rod_profile_table.to_csv(rod_profile_csv, index=False)
marker_csv = OUT_DIR / f"{ROD_PROFILE_MARKER_STEM}.csv"
marker_table.to_csv(marker_csv, index=False)
print(f"saved={rod_profile_csv}")
print(f"saved={marker_csv}")

support_diagnostic_stem = f"{ROD_PROFILE_STEM}_support_diagnostics"
fig, support_axes = plt.subplots(2, 2, figsize=(JOURNAL_FULL_WIDTH_IN, 4.8), constrained_layout=True)
for ax, metric, ylabel in zip(
    support_axes.ravel(),
    ("pixel_count", "acceptance_sum", "background_sum", "background_density"),
    ("Pixel count", "Acceptance sum", "Raw background sum", "Background density"),
):
    for (m_value, branch_value), sub in rod_profile_table.groupby(["m", "branch"]):
        sub = sub[np.asarray(sub["pixel_count"], dtype=np.float64) > 0].copy()
        if sub.empty or metric not in sub:
            continue
        x = np.asarray(sub["qz_center"], dtype=np.float64)
        y = np.asarray(sub[metric], dtype=np.float64)
        order = np.argsort(x)
        ax.plot(x[order], y[order], linewidth=0.7, alpha=0.65, label=f"m={int(m_value)} {branch_value}")
    ax.set_xlabel(r"$Q_z$ ($\mathrm{\AA}^{-1}$)")
    ax.set_ylabel(ylabel)
    ax.grid(True, color=JOURNAL_GRID_COLOR, linewidth=0.45)
    finish_axes(ax)
support_axes[0, 0].legend(loc="best", fontsize=5.2, frameon=False, ncol=2)
support_png, support_pdf = save_manuscript_figure(fig, support_diagnostic_stem)
plt.close(fig)
print(f"saved={support_png}")
print(f"saved={support_pdf}")

rod_profile_note = OUT_DIR / f"{ROD_PROFILE_STEM}.md"
rod_note_lines = [
    "# Bi2Se3 5° $Q_r$-rod $Q_z$ profiles",
    "",
    f"Source state: `{STATE_PATH}`",
    f"Tilt used for Q conversion: `{ROD_PROFILE_TILT_DEG}°`",
    f"Delta $Q_r$: `{qr_rod_delta_qr:.4g}` Å^-1",
    f"Displayed/integrated caked support is limited to `2theta <= {ROD_PROFILE_MAX_TWO_THETA_DEG:.1f}°`.",
    "Phi windows: `-90° < phi < 0°` and `0° < phi < 90°`.",
    "",
    "The figure uses only the 5° background. Non-specular traces are integrated directly in detector Qr/Qz space.",
    "For each Qz bin, detector pixels are selected by Qr0 +/- delta_Qr, branch phi window, and the 2theta display limit before summing intensity.",
    "The selected-$Q_r$ caked masks are retained as display overlays, not as the numerical integrator.",
    "The plotted traces are acceptance-normalized intensity densities. Raw summed columns are retained for audit only.",
    "When caked sum fields are available, density uses sum_signal / sum_normalization. Otherwise it falls back to acceptance weights, then pixel_count.",
    "The measured trace is the density profile after subtracting a rolling lower-envelope baseline.",
    "It is compared with a joint $Q_z$ line fit: all projected branch-point peaks in a rod/branch are fit simultaneously as a sum of Gaussian peaks.",
    "The detector-space fit model remains in the CSV as `fit_density`; the plotted line uses `joint_fit_density`, which avoids double-counting overlap between close peaks.",
    "Titles are $Q_r$ values; marker tick labels are projected $Q_z$ values for the fitted branch points.",
    r"$Q_r$ branch masks extend to the projected $Q_z$ endpoint nearest $\phi = \pm90^\circ$.",
    r"The caked-region figure also includes the $H=K=0$ strip for $-10^\circ < \phi < 10^\circ$ and $5^\circ < 2\theta < 25^\circ$.",
    "",
    "## Rods",
    "",
    "| m | $Q_r$ (Å^-1) | marker count |",
    "|---:|---:|---:|",
]
for rod in rod_entries:
    marker_count = int((marker_table["m"] == int(rod["m"])).sum()) if not marker_table.empty else 0
    rod_note_lines.append(f"| {int(rod['m'])} | {float(rod['qr']):.6f} | {marker_count} |")
rod_note_lines.append("| 0 | 0.000000 | H=K=0 Qz rod, -10 deg < phi < 10 deg |")
rod_profile_note.write_text("\n".join(rod_note_lines) + "\n", encoding="utf-8")
print(f"saved={rod_profile_note}")
if profile_failures:
    print("profile notes:")
    for item in profile_failures:
        print(" -", item)

plot_rod_entries = rod_entries + [specular_rod_entry]
fig, axes = plt.subplots(
    len(plot_rod_entries),
    len(phi_windows),
    figsize=(JOURNAL_FULL_WIDTH_IN, max(2.05 * len(plot_rod_entries), 5.0)),
    squeeze=False,
    sharex=False,
    sharey=False,
    constrained_layout=True,
)
for row, rod in enumerate(plot_rod_entries):
    if int(rod["m"]) == 0:
        ax = axes[row, 0]
        sub = rod_profile_table[(rod_profile_table["m"] == 0) & (rod_profile_table["branch"] == "qz") & (rod_profile_table["pixel_count"] > 0)].copy()
        if not sub.empty:
            measured_norm, fitted_norm = normalized_profile_pair(sub["background_density"], sub.get("joint_fit_density", sub["fit_density"]))
            x = np.asarray(sub["qz_center"], dtype=np.float64)
            order = np.argsort(x)
            ax.plot(x[order], measured_norm[order], color=JOURNAL_DATA_COLOR, linewidth=1.0, alpha=0.92, label="Measured, baseline subtracted")
            ax.plot(x[order], fitted_norm[order], color=JOURNAL_FIT_COLOR, linewidth=1.0, alpha=0.96, linestyle="--", label="Fitted Gaussian peaks")
        ax.axhline(0.0, color="0.80", linewidth=0.45)
        ax.grid(True, color=JOURNAL_GRID_COLOR, linewidth=0.45)
        ax.set_title(r"$Q_r=0.000\,\mathrm{\AA}^{-1}$, $H=K=0$ $Q_z$ rod")
        ax.set_xlabel(r"$Q_z$ ($\mathrm{\AA}^{-1}$), $\theta_i=5^\circ$")
        ax.set_ylabel("Normalized intensity density")
        ax.tick_params(length=2.0, width=0.45, labelsize=6.1, direction="in", top=True, right=True)
        for spine in ax.spines.values():
            spine.set_linewidth(0.55)
        axes[row, 1].axis("off")
        continue
    for col, (branch_name, _phi_min, _phi_max, branch_label) in enumerate(phi_windows):
        ax = axes[row, col]
        sub = rod_profile_table[
            (rod_profile_table["m"] == int(rod["m"]))
            & (rod_profile_table["branch"] == branch_name)
            & (rod_profile_table["pixel_count"] > 0)
        ].copy()
        if not sub.empty:
            measured_norm, fitted_norm = normalized_profile_pair(sub["background_density"], sub.get("joint_fit_density", sub["fit_density"]))
            x = np.asarray(sub["qz_center"], dtype=np.float64)
            order = np.argsort(x)
            x = x[order]
            measured_norm = measured_norm[order]
            fitted_norm = fitted_norm[order]
            ax.plot(x, measured_norm, color=JOURNAL_DATA_COLOR, linewidth=1.0, alpha=0.92, label="Measured, baseline subtracted")
            ax.plot(x, fitted_norm, color=JOURNAL_FIT_COLOR, linewidth=1.0, alpha=0.96, linestyle="--", label="Fitted Gaussian peaks")

            mark = marker_table[
                (marker_table["m"] == int(rod["m"]))
                & (marker_table["branch"] == branch_name)
            ].copy() if not marker_table.empty else pd.DataFrame()
            if not mark.empty:
                qz_markers = np.asarray(mark["qz_marker"], dtype=np.float64)
                y_markers = np.interp(qz_markers, x, measured_norm, left=np.nan, right=np.nan)
                finite_markers = np.isfinite(qz_markers) & np.isfinite(y_markers)
                ax.scatter(
                    qz_markers[finite_markers],
                    y_markers[finite_markers],
                    s=40,
                    facecolors="white",
                    edgecolors="black",
                    linewidths=1.0,
                    zorder=5,
                )
                for qz_value in qz_markers[finite_markers]:
                    ax.axvline(float(qz_value), color="0.60", linewidth=0.55, linestyle=":", zorder=1)
                ax.set_xticks(qz_markers[finite_markers])
                ax.set_xticklabels([f"{value:.2f}" for value in qz_markers[finite_markers]], rotation=90, fontsize=6.6)
        ax.axhline(0.0, color="0.80", linewidth=0.45)
        ax.grid(True, color=JOURNAL_GRID_COLOR, linewidth=0.45)
        ax.set_title(rf"$Q_r={float(rod['qr']):.3f}\,\mathrm{{\AA}}^{{-1}}$, {branch_label}")
        ax.set_xlabel(r"$Q_z$ ($\mathrm{\AA}^{-1}$), $\theta_i=5^\circ$")
        if col == 0:
            ax.set_ylabel("Normalized intensity density")
        ax.tick_params(length=2.0, width=0.45, labelsize=6.1, direction="in", top=True, right=True)
        for spine in ax.spines.values():
            spine.set_linewidth(0.55)

legend_handles = [
    axes[0, 0].plot([], [], color=JOURNAL_DATA_COLOR, linewidth=1.0, label="Measured, baseline subtracted")[0],
    axes[0, 0].plot([], [], color=JOURNAL_FIT_COLOR, linewidth=1.0, linestyle="--", label="Fitted Gaussian peaks")[0],
    axes[0, 0].scatter([], [], s=40, facecolors="white", edgecolors="black", linewidths=1.0, label="Projected branch point"),
]
fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(rf"{SAMPLE_LABEL}, $\theta_i=5^\circ$: $Q_r$-rod profiles with projected $Q_z$ branch points", y=1.045)
rod_profile_png, rod_profile_pdf = save_manuscript_figure(fig, ROD_PROFILE_STEM)
plt.close(fig)
print(f"saved={rod_profile_png}")
print(f"saved={rod_profile_pdf}")
display(Image(filename=str(rod_profile_png)))

caked_region_theta_min = float(np.asarray(profile_bg["theta_axis"], dtype=np.float64)[0])
caked_region_theta_max = min(
    float(np.asarray(profile_bg["theta_axis"], dtype=np.float64)[-1]),
    float(ROD_PROFILE_MAX_TWO_THETA_DEG),
)
caked_region_extent = [
    float(np.asarray(profile_bg["theta_axis"], dtype=np.float64)[0]),
    float(np.asarray(profile_bg["theta_axis"], dtype=np.float64)[-1]),
    float(np.asarray(profile_bg["phi_axis"], dtype=np.float64)[0]),
    float(np.asarray(profile_bg["phi_axis"], dtype=np.float64)[-1]),
]
theta_region_axis = np.asarray(profile_bg["theta_axis"], dtype=np.float64)
phi_region_axis = np.asarray(profile_bg["phi_axis"], dtype=np.float64)
specular_region_mask = (
    (theta_region_axis[None, :] >= 5.0)
    & (theta_region_axis[None, :] <= 25.0)
    & (phi_region_axis[:, None] > -10.0)
    & (phi_region_axis[:, None] < 10.0)
)
specular_region_points = []
for item in profile_bg["fit_results"]:
    try:
        h, k, _l = parse_hkl_label(str(item["label"]))
    except Exception:
        continue
    theta0 = float(item["params"][1])
    phi0 = float(item["params"][2])
    if h == 0 and k == 0 and 5.0 <= theta0 <= 25.0 and -10.0 < phi0 < 10.0:
        specular_region_points.append((theta0, phi0))
profile_caked_display = np.asarray(profile_bg.get("caked_display_image", profile_bg.get("caked_density_image", profile_bg["caked_image"])), dtype=np.float64)
profile_caked_mode_label = "raw accumulated intensity" if profile_bg.get("caked_intensity_mode") == "raw_sum" else "intensity density"
caked_region_bg = caked_log_image(profile_caked_display)
region_vmin, region_vmax = robust_display_limits(caked_region_bg, low=1.0, high=99.7)
region_colors = JOURNAL_REGION_COLORS

fig, ax = plt.subplots(figsize=(JOURNAL_FULL_WIDTH_IN, 4.2), constrained_layout=True)
ax.imshow(
    caked_region_bg,
    extent=caked_region_extent,
    origin="lower",
    aspect="auto",
    cmap="gray_r",
    vmin=region_vmin,
    vmax=region_vmax,
)
legend_handles = []
for index, rod in enumerate(rod_entries):
    color = region_colors[index % len(region_colors)]
    legend_handles.append(plt.Line2D([], [], color=color, linewidth=1.2, label=rf"$Q_r={float(rod['qr']):.3f}\,\mathrm{{\AA}}^{{-1}}$"))
    for branch_name, linestyle in (("-", "-"), ("+", "--")):
        overlays = [
            item for item in region_overlays
            if int(item["m"]) == int(rod["m"]) and str(item["branch"]) == branch_name
        ]
        for item in overlays:
            mask = np.asarray(item["mask"], dtype=bool)
            region_fill = np.ma.masked_where(~mask, np.ones_like(mask, dtype=np.float64))
            ax.imshow(
                region_fill,
                extent=caked_region_extent,
                origin="lower",
                aspect="auto",
                cmap=ListedColormap([color]),
                alpha=0.16,
                interpolation="nearest",
            )
            ax.contour(
                mask.astype(np.float64),
                levels=[0.5],
                extent=caked_region_extent,
                origin="lower",
                colors=[color],
                linewidths=0.85,
                linestyles=linestyle,
            )
        mark = marker_table[
            (marker_table["m"] == int(rod["m"]))
            & (marker_table["branch"] == branch_name)
        ].copy() if not marker_table.empty else pd.DataFrame()
        if not mark.empty:
            mark = mark[
                np.asarray(mark["refined_two_theta_deg"], dtype=np.float64)
                <= float(ROD_PROFILE_MAX_TWO_THETA_DEG)
            ].copy()
        if not mark.empty:
            ax.scatter(
                np.asarray(mark["refined_two_theta_deg"], dtype=np.float64),
                np.asarray(mark["refined_phi_deg"], dtype=np.float64),
                s=22,
                facecolors="white",
                edgecolors="black",
                linewidths=0.7,
                zorder=6,
            )

specular_color = OKABE_ITO["purple"]
specular_fill = np.ma.masked_where(~specular_region_mask, np.ones_like(specular_region_mask, dtype=np.float64))
ax.imshow(
    specular_fill,
    extent=caked_region_extent,
    origin="lower",
    aspect="auto",
    cmap=ListedColormap([specular_color]),
    alpha=0.32,
    interpolation="nearest",
)
ax.contour(
    specular_region_mask.astype(np.float64),
    levels=[0.5],
    extent=caked_region_extent,
    origin="lower",
    colors=[specular_color],
    linewidths=1.6,
)
ax.text(
    6.0,
    8.0,
    r"$H=K=0$ $Q_z$ rod",
    color=specular_color,
    fontsize=7.2,
    ha="left",
    va="bottom",
    bbox={"facecolor": "white", "edgecolor": specular_color, "linewidth": 0.6, "alpha": 0.82, "pad": 1.8},
)
if specular_region_points:
    specular_points = np.asarray(specular_region_points, dtype=np.float64)
    ax.scatter(
        specular_points[:, 0],
        specular_points[:, 1],
        s=22,
        facecolors="white",
        edgecolors="black",
        linewidths=0.7,
        zorder=6,
    )
legend_handles.append(plt.Line2D([], [], color=specular_color, linewidth=1.2, label=r"$H=K=0$, $5^\circ<2\theta<25^\circ$"))

branch_handles = [
    plt.Line2D([], [], color="0.1", linewidth=0.9, linestyle="-", label=r"$-90^\circ<\phi<0^\circ$"),
    plt.Line2D([], [], color="0.1", linewidth=0.9, linestyle="--", label=r"$0^\circ<\phi<90^\circ$"),
    plt.Line2D([], [], marker="o", color="none", markerfacecolor="white", markeredgecolor="black", markersize=4.5, label="Projected branch point"),
]
ax.legend(handles=legend_handles + branch_handles, loc="upper right", frameon=True, framealpha=0.88, fontsize=6.3)
ax.set_title(rf"{SAMPLE_LABEL}, $\theta_i=5^\circ$: selected $Q_r$ integration regions ({profile_caked_mode_label})")
ax.set_xlabel(r"$2\theta$ (°)")
ax.set_ylabel(r"$\phi$ (°)")
finish_axes(ax)
ax.set_xlim(caked_region_theta_min, caked_region_theta_max)
caked_region_png, caked_region_pdf = save_manuscript_figure(fig, ROD_PROFILE_REGION_STEM)
plt.close(fig)
print(f"saved={caked_region_png}")
print(f"saved={caked_region_pdf}")
display(Image(filename=str(caked_region_png)))
